# Qwen-Image 2.1 PixelHead v0.2（Colab A100 全分辨率感知锐化升级台）

针对 `v0.1` 在 **几像素高的小字（如车票/路牌细字）** 与 **近景人像微距肤质/睫毛** 上比 `644 MB` 官方 VAE 偏软的问题（[Issue #1](https://github.com/Wither033/ComfyUI-QwenImage21-PixelHead/issues/1)），本 Notebook 使用 **Google Colab A100 GPU (40GB / 80GB)** 进行四项核心升级：

1. **四级渐进上采样至 `1×1` 原生像素级卷积 (`QwenImage21PixelHeadV2`)**：
   - 从 `v0.1` 的 `16px → 8px` 线性展开升级为 **`16px → 8px → 4px → 2px → 1px` 全分辨率深度可分离 `3×3` 残差细化金字塔**（`FP32` 末层输出），消除 `8×8` 线性块对 `1～2 px` 笔画与睫毛的表达瓶颈；
   - 新增 **`SigmaAdaLN` 末步噪声条件调制**，让 `8-Step Turbo`（$\sigma \approx 0.22$）与 `20～30 步 Base CFG=3.5`（$\sigma \approx 0.05$）各自精准适配，互不折中；
   - **零损热启动（Warm-Start from `v0.1`）**：直接复用 `v0.1` 主干与 `8×8` 锚点分支，新细化金字塔零初始化，**第 0 步即继承 `33.9 dB` 基线**，全部算力集中用于雕刻高频细节！
2. **真机 `1024×1024` 多步采样轨迹蒸馏（含真实 `CFG = 3.5` 特征外推）**：
   - 直接在 A100 上跑真实 `KSampler`（50% `8步 Turbo CFG=1.0` + 50% `20步 Base CFG=3.5`），覆盖高清近景人像、手持小字车票/菜单/标牌、二次元细线稿，彻底消除 `v0.1` 未见过 `CFG > 1` 组合特征的分布偏差；
   - 安装 `fonts-noto-cjk` 思源黑体/宋体，合成 `9px～26px` 中英双语密集小字校准卡。
3. **去模糊复合感知损失（`Charbonnier L1 + 边缘加权多尺度梯度 + LPIPS-VGG + 2D 高频 FFT`）**：
   - 将 `MSE` 权重从 `5.0` 大幅下调至 `0.4`，引入 `LPIPS-VGG` 五层特征感知损失与 2D 傅里叶高频幅度约束，并配合 `EMA (0.999)` 平滑权重。
4. **三方同屏盲测（`Official 644MB VAE` vs `v0.1` vs `v0.2`）**：
   - 自动在留出集（含 Issue #1 同款小字车票与特写人像）对比 `PSNR`、`LPIPS-VGG`、边缘锐度保持率与解码耗时，并导出 `1:1` 三列对比大图与 `qwen_image_2.1_pixel_head_v02_bf16.safetensors` 至 Google Drive。

### 运行方式
右上角确认运行时为 **`A100 GPU`** ➔ 点击顶部菜单 **`Runtime (代码执行程序) ➔ Run all (全部运行)`**。

In [ ]:
# @title 1. 挂载 Google Drive、校验 A100 显卡并部署 v0.2 插件代码
import json
import os
import subprocess
from pathlib import Path
import torch
from google.colab import drive

subprocess.run(["nvidia-smi"], check=False)
assert torch.cuda.is_available(), "未检测到 CUDA GPU，请切换运行时为 A100 GPU！"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ 当前 GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

drive.mount("/content/drive")

DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
PROJECT_V2 = Path("/content/drive/MyDrive/FastFrames_QwenImage21_v02")
CKPT_V2 = PROJECT_V2 / "checkpoints"
EVAL_V2 = PROJECT_V2 / "eval_report"
PIXEL_HEADS_DRIVE = DRIVE_HUB / "models/pixel_heads"
VAE_DRIVE = DRIVE_HUB / "models/vae"
CUSTOM_NODE_DRIVE = DRIVE_HUB / "custom_nodes_qwenimage21/ComfyUI-QwenImage21-PixelHead"
LOCAL_SRC = Path("/content/ComfyUI-QwenImage21-PixelHead")

for d in (CKPT_V2, EVAL_V2, PIXEL_HEADS_DRIVE, VAE_DRIVE, CUSTOM_NODE_DRIVE, LOCAL_SRC):
    d.mkdir(parents=True, exist_ok=True)

FILES = {
  "qwen21_pixel_head.py": "\"\"\"PyTorch implementation (adapted from sorryhyun/anima_lora & FastFrames) of Direct Pixel Readout Head for Qwen-Image 2.1 (v0.1 & v0.2).\n\nLicense: MIT\nSupports:\n  - `QwenImage21PixelHead` (`v0.1`, 16px -> 8px + 8x8 linear unpatchify, ~5.5 ms)\n  - `QwenImage21PixelHeadX2` (`v0.1` 2x super-resolution branch, 1024 -> 2048)\n  - `QwenImage21PixelHeadV2` (`v0.2` High-Fidelity Edition: explicit `sigma` AdaLN conditioning +\n    warm-startable 16px/8px ConvNeXt trunk + 8x8 coarse anchor + progressive 4px -> 2px -> 1px\n    full-resolution depthwise-separable refinement pyramid for crisp small text & portrait pores)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nTOKEN_PIXELS = 16  # Qwen-Image 2.1 VAE spatial downscale is 16x, DiT patch is 1x1\n\n\nclass ConvNeXtBlock2d(nn.Module):\n    \"\"\"2D ConvNeXt block with 3x3 depthwise conv, channel LayerNorm, 4x GELU MLP, and LayerScale.\"\"\"\n\n    def __init__(self, dim: int, mlp_ratio: int = 4, ls_init: float = 1.0, zero_out: bool = False):\n        super().__init__()\n        self.dw = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)\n        self.norm = nn.LayerNorm(dim, eps=1e-6)\n        hidden = dim * mlp_ratio\n        self.fc1 = nn.Linear(dim, hidden)\n        self.fc2 = nn.Linear(hidden, dim)\n        self.gamma = nn.Parameter(torch.full((dim,), float(ls_init)))\n        if zero_out:\n            nn.init.zeros_(self.fc2.weight)\n            nn.init.zeros_(self.fc2.bias)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # x: [B, C, H, W]\n        h = self.dw(x).movedim(1, -1)  # [B, H, W, C]\n        h = self.fc2(F.gelu(self.fc1(self.norm(h))))\n        h = (h * self.gamma).movedim(-1, 1)\n        return x + h\n\n\nclass PixelRefineBlock2d(nn.Module):\n    \"\"\"Lightweight high-resolution residual depthwise-separable 3x3 block for 2px and 1px stages.\"\"\"\n\n    def __init__(self, dim: int, expand: int = 2):\n        super().__init__()\n        hidden = dim * expand\n        self.dw = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)\n        self.norm = nn.GroupNorm(num_groups=min(8, dim), num_channels=dim, eps=1e-6)\n        self.pw1 = nn.Conv2d(dim, hidden, kernel_size=1)\n        self.pw2 = nn.Conv2d(hidden, dim, kernel_size=1)\n        nn.init.zeros_(self.pw2.weight)\n        nn.init.zeros_(self.pw2.bias)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        h = self.pw2(F.silu(self.pw1(self.norm(self.dw(x)))))\n        return x + h\n\n\nclass SigmaAdaLN(nn.Module):\n    \"\"\"Sinusoidal log-sigma embedding + zero-initialized AdaLN (scale, shift) modulation.\"\"\"\n\n    def __init__(self, embed_dim: int = 256, target_dims: tuple[int, ...] = (1024, 512)):\n        super().__init__()\n        self.embed_dim = int(embed_dim)\n        self.mlp = nn.Sequential(\n            nn.Linear(embed_dim, embed_dim),\n            nn.SiLU(),\n            nn.Linear(embed_dim, embed_dim),\n            nn.SiLU(),\n        )\n        self.heads = nn.ModuleList([nn.Linear(embed_dim, 2 * d) for d in target_dims])\n        for hd in self.heads:\n            nn.init.zeros_(hd.weight)\n            nn.init.zeros_(hd.bias)\n\n    def embed_sigma(self, sigma: torch.Tensor | float, batch: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:\n        if not torch.is_tensor(sigma):\n            s = torch.full((batch,), float(sigma), device=device, dtype=torch.float32)\n        else:\n            s = sigma.to(device=device, dtype=torch.float32).reshape(-1)\n            if s.numel() == 1 and batch > 1:\n                s = s.expand(batch)\n        log_s = torch.log(s.clamp(min=1e-4, max=2.0))  # [-9.2, 0.7]\n        half = self.embed_dim // 2\n        freqs = torch.exp(-math.log(1000.0) * torch.arange(half, device=device, dtype=torch.float32) / max(1, half - 1))\n        ang = log_s[:, None] * freqs[None, :]\n        emb = torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1).to(dtype=dtype)\n        return self.mlp(emb)\n\n    def modulate_nchw(self, x: torch.Tensor, cond: torch.Tensor, head_idx: int) -> torch.Tensor:\n        scale, shift = self.heads[head_idx](cond).chunk(2, dim=-1)\n        return x * (1.0 + scale[:, :, None, None]) + shift[:, :, None, None]\n\n\ndef unpatchify_2d(tokens: torch.Tensor, patch: int, out_channels: int) -> torch.Tensor:\n    \"\"\"Convert `[B, gh, gw, patch * patch * C]` -> `[B, C, gh * patch, gw * patch]`.\"\"\"\n    b, gh, gw, _ = tokens.shape\n    x = tokens.view(b, gh, gw, patch, patch, out_channels)\n    x = x.permute(0, 5, 1, 3, 2, 4).contiguous()\n    return x.view(b, out_channels, gh * patch, gw * patch)\n\n\nclass QwenImage21PixelHead(nn.Module):\n    \"\"\"v0.1 1x Direct Pixel Head for Qwen-Image 2.1 (`16x16` RGBA/RGB pixels per DiT token).\"\"\"\n\n    def __init__(\n        self,\n        d_feat: int = 4096,\n        d_lat: int = 64,\n        width: int = 1024,\n        depth_low: int = 2,\n        depth_high: int = 1,\n        out_channels: int = 4,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        if width % 2 != 0:\n            raise ValueError(f\"width must be even, got {width}\")\n        self.d_feat = int(d_feat)\n        self.d_lat = int(d_lat)\n        self.width = int(width)\n        self.depth_low = int(depth_low)\n        self.depth_high = int(depth_high)\n        self.out_channels = int(out_channels)\n        self.ls_init = float(ls_init)\n\n        w_high = width // 2\n        self.out_patch = TOKEN_PIXELS // 2  # 8x8 pixels per 2x-upsampled position\n\n        self.feat_norm = nn.LayerNorm(self.d_feat, eps=1e-6)\n        self.feat_proj = nn.Linear(self.d_feat, width, bias=False)\n        self.lat_proj = nn.Linear(self.d_lat, width, bias=True)\n        self.entry_norm = nn.LayerNorm(width, eps=1e-6)\n\n        self.low_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(width, ls_init=ls_init) for _ in range(depth_low)]\n        )\n        self.up_norm = nn.LayerNorm(width, eps=1e-6)\n        self.up_proj = nn.Linear(width, 4 * w_high)\n        self.high_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(w_high, ls_init=ls_init) for _ in range(depth_high)]\n        )\n        self.norm_out = nn.LayerNorm(w_high, eps=1e-6)\n        self.out_proj = nn.Linear(w_high, self.out_patch * self.out_patch * self.out_channels)\n        nn.init.normal_(self.out_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out_proj.bias)\n\n        self.meta: dict[str, str] = {}\n\n    def forward_trunk(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor) -> torch.Tensor:\n        b, _, gh, gw = x0_hat_norm.shape\n        if feat.ndim == 3:\n            feat = feat.view(b, gh, gw, self.d_feat)\n        lat_tok = x0_hat_norm.movedim(1, -1)  # [B, gh, gw, d_lat]\n\n        h = self.entry_norm(self.feat_proj(self.feat_norm(feat)) + self.lat_proj(lat_tok))\n        h = h.movedim(-1, 1).contiguous()  # [B, width, gh, gw]\n        for blk in self.low_blocks:\n            h = blk(h)\n\n        u = self.up_proj(self.up_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        h = F.pixel_shuffle(u, 2)  # [B, width//2, 2*gh, 2*gw]\n        for blk in self.high_blocks:\n            h = blk(h)\n        return h\n\n    def forward(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor) -> torch.Tensor:\n        h = self.forward_trunk(feat, x0_hat_norm)\n        h_tok = self.norm_out(h.movedim(1, -1))\n        with torch.autocast(device_type=h_tok.device.type, enabled=False):\n            patches = self.out_proj(h_tok.float())\n        return unpatchify_2d(patches, self.out_patch, self.out_channels)\n\n\nclass QwenImage21PixelHeadV2(nn.Module):\n    \"\"\"v0.2 High-Fidelity Direct Pixel Head (`16px -> 8px -> 4px -> 2px -> 1px` full-resolution pyramid).\n\n    - Warm-starts cleanly from `QwenImage21PixelHead` (`v0.1`) so step 0 already achieves ~33.9 dB!\n    - Adds `SigmaAdaLN` so 8-step Turbo (`sigma~0.22`) and 25-step Base (`sigma~0.05`) never compromise each other.\n    - Adds progressive `4px -> 2px -> 1px` ConvNeXt + depthwise-separable 3x3 pixel refinement convs\n      plus an FP32 `3x3` full-resolution residual head to recover 1-2px small text & portrait skin/eyelash detail.\n    \"\"\"\n\n    def __init__(\n        self,\n        base: Optional[QwenImage21PixelHead] = None,\n        d_feat: int = 4096,\n        d_lat: int = 64,\n        width: int = 1024,\n        depth_low: int = 2,\n        depth_high: int = 1,\n        extra_low: int = 1,\n        extra_high: int = 2,\n        depth_4px: int = 2,\n        depth_2px: int = 2,\n        depth_1px: int = 2,\n        c_4px: int = 256,\n        c_2px: int = 128,\n        c_1px: int = 64,\n        out_channels: int = 4,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        self.base = base if base is not None else QwenImage21PixelHead(\n            d_feat=d_feat,\n            d_lat=d_lat,\n            width=width,\n            depth_low=depth_low,\n            depth_high=depth_high,\n            out_channels=out_channels,\n            ls_init=ls_init,\n        )\n        w0 = self.base.width\n        w8 = w0 // 2\n        self.extra_low = int(extra_low)\n        self.extra_high = int(extra_high)\n        self.depth_4px = int(depth_4px)\n        self.depth_2px = int(depth_2px)\n        self.depth_1px = int(depth_1px)\n        self.c_4px = int(c_4px)\n        self.c_2px = int(c_2px)\n        self.c_1px = int(c_1px)\n        self.out_channels = self.base.out_channels\n\n        self.sigma_adaln = SigmaAdaLN(embed_dim=256, target_dims=(w0, w8, c_4px))\n\n        # Extra zero-initialized residual ConvNeXt capacity on 16px and 8px grids\n        self.low_ext = nn.ModuleList([ConvNeXtBlock2d(w0, ls_init=ls_init, zero_out=True) for _ in range(self.extra_low)])\n        self.high_ext = nn.ModuleList([ConvNeXtBlock2d(w8, ls_init=ls_init, zero_out=True) for _ in range(self.extra_high)])\n\n        # 8px -> 4px stage\n        self.up4_norm = nn.LayerNorm(w8, eps=1e-6)\n        self.up4_proj = nn.Linear(w8, 4 * self.c_4px)\n        self.blocks_4px = nn.ModuleList([ConvNeXtBlock2d(self.c_4px, ls_init=ls_init) for _ in range(self.depth_4px)])\n\n        # 4px -> 2px stage\n        self.up2_norm = nn.LayerNorm(self.c_4px, eps=1e-6)\n        self.up2_proj = nn.Linear(self.c_4px, 4 * self.c_2px)\n        self.blocks_2px = nn.ModuleList([PixelRefineBlock2d(self.c_2px, expand=2) for _ in range(self.depth_2px)])\n\n        # 2px -> 1px FULL NATIVE PIXEL RESOLUTION stage (e.g. 1024x1024)\n        self.up1_conv = nn.Conv2d(self.c_2px, 4 * self.c_1px, kernel_size=1)\n        self.blocks_1px = nn.ModuleList([PixelRefineBlock2d(self.c_1px, expand=2) for _ in range(self.depth_1px)])\n        self.detail_norm = nn.GroupNorm(num_groups=min(8, self.c_1px), num_channels=self.c_1px, eps=1e-6)\n        # Zero-init FP32 3x3 output conv so step 0 exactly preserves warm-started v0.1 anchor!\n        self.detail_out = nn.Conv2d(self.c_1px, self.out_channels, kernel_size=3, padding=1)\n        nn.init.zeros_(self.detail_out.weight)\n        nn.init.zeros_(self.detail_out.bias)\n\n        self.meta: dict[str, str] = {}\n\n    def forward(\n        self,\n        feat: torch.Tensor,\n        x0_hat_norm: torch.Tensor,\n        sigma: torch.Tensor | float = 0.10,\n    ) -> torch.Tensor:\n        bsz, _, gh, gw = x0_hat_norm.shape\n        b = self.base\n        if feat.ndim == 3:\n            feat = feat.view(bsz, gh, gw, b.d_feat)\n        lat_tok = x0_hat_norm.movedim(1, -1)\n\n        s_cond = self.sigma_adaln.embed_sigma(sigma, bsz, x0_hat_norm.device, feat.dtype)\n\n        # 16px stage\n        h = b.entry_norm(b.feat_proj(b.feat_norm(feat)) + b.lat_proj(lat_tok))\n        h = h.movedim(-1, 1).contiguous()\n        h = self.sigma_adaln.modulate_nchw(h, s_cond, 0)\n        for blk in b.low_blocks:\n            h = blk(h)\n        for blk in self.low_ext:\n            h = blk(h)\n\n        # 8px stage\n        u8 = b.up_proj(b.up_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        h8 = F.pixel_shuffle(u8, 2)\n        h8 = self.sigma_adaln.modulate_nchw(h8, s_cond, 1)\n        for blk in b.high_blocks:\n            h8 = blk(h8)\n        for blk in self.high_ext:\n            h8 = blk(h8)\n\n        # Coarse 8x8 FP32 anchor (locks global color & gradients immediately)\n        h8_tok = b.norm_out(h8.movedim(1, -1))\n        with torch.autocast(device_type=h8_tok.device.type, enabled=False):\n            anchor_patches = b.out_proj(h8_tok.float())\n        anchor_img = unpatchify_2d(anchor_patches, b.out_patch, self.out_channels)  # fp32 [B, 4, H, W]\n\n        # Progressive high-frequency detail pyramid: 8px -> 4px -> 2px -> 1px\n        u4 = self.up4_proj(self.up4_norm(h8.movedim(1, -1))).movedim(-1, 1).contiguous()\n        h4 = F.pixel_shuffle(u4, 2)\n        h4 = self.sigma_adaln.modulate_nchw(h4, s_cond, 2)\n        for blk in self.blocks_4px:\n            h4 = blk(h4)\n\n        u2 = self.up2_proj(self.up2_norm(h4.movedim(1, -1))).movedim(-1, 1).contiguous()\n        h2 = F.pixel_shuffle(u2, 2)\n        for blk in self.blocks_2px:\n            h2 = blk(h2)\n\n        h1 = F.pixel_shuffle(F.silu(self.up1_conv(h2)), 2)\n        for blk in self.blocks_1px:\n            h1 = blk(h1)\n\n        with torch.autocast(device_type=h1.device.type, enabled=False):\n            h1_fp32 = F.silu(self.detail_norm.float()(h1.float()))\n            detail_img = self.detail_out(h1_fp32)\n\n        return anchor_img + detail_img\n\n\nclass QwenImage21PixelHeadX2(nn.Module):\n    \"\"\"Optional 2x Super-Resolution Direct Pixel Head (`32x32` pixels per DiT token).\"\"\"\n\n    def __init__(\n        self,\n        base: QwenImage21PixelHead,\n        depth_x2: int = 2,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        self.base = base\n        self.depth_x2 = int(depth_x2)\n        w_in = base.width // 2\n        w_x2 = w_in // 2\n        self.out_channels = base.out_channels\n        self.out_patch = base.out_patch\n\n        self.up2_norm = nn.LayerNorm(w_in, eps=1e-6)\n        self.up2_proj = nn.Linear(w_in, 4 * w_x2)\n        self.x2_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(w_x2, ls_init=ls_init) for _ in range(depth_x2)]\n        )\n        self.norm2 = nn.LayerNorm(w_x2, eps=1e-6)\n        self.out2_proj = nn.Linear(w_x2, self.out_patch * self.out_patch * self.out_channels)\n        nn.init.normal_(self.out2_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out2_proj.bias)\n        self.meta: dict[str, str] = {}\n\n    def forward(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor, scale: int = 2) -> torch.Tensor:\n        h = self.base.forward_trunk(feat, x0_hat_norm)\n        if scale == 1:\n            h1 = self.base.norm_out(h.movedim(1, -1))\n            with torch.autocast(device_type=h1.device.type, enabled=False):\n                p1 = self.base.out_proj(h1.float())\n            return unpatchify_2d(p1, self.base.out_patch, self.out_channels)\n\n        u2 = self.up2_proj(self.up2_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        g = F.pixel_shuffle(u2, 2)\n        for blk in self.x2_blocks:\n            g = blk(g)\n        h2 = self.norm2(g.movedim(1, -1))\n        with torch.autocast(device_type=h2.device.type, enabled=False):\n            p2 = self.out2_proj(h2.float())\n        return unpatchify_2d(p2, self.out_patch, self.out_channels)\n\n\ndef save_qwen21_pixel_head(\n    head: QwenImage21PixelHead | QwenImage21PixelHeadX2 | QwenImage21PixelHeadV2,\n    path: str | Path,\n    extra_meta: Optional[dict[str, str]] = None,\n    save_bf16_trunk: bool = False,\n) -> None:\n    from safetensors.torch import save_file\n\n    is_v2 = isinstance(head, QwenImage21PixelHeadV2)\n    is_x2 = isinstance(head, QwenImage21PixelHeadX2)\n    base = head.base if (is_v2 or is_x2) else head\n\n    if is_v2:\n        arch = \"qwen_image21_pixel_head_v2\"\n    elif is_x2:\n        arch = \"qwen_image21_pixel_head_x2\"\n    else:\n        arch = \"qwen_image21_pixel_head\"\n\n    meta = {\n        \"arch\": arch,\n        \"d_feat\": str(base.d_feat),\n        \"d_lat\": str(base.d_lat),\n        \"width\": str(base.width),\n        \"depth_low\": str(base.depth_low),\n        \"depth_high\": str(base.depth_high),\n        \"out_channels\": str(base.out_channels),\n        \"ls_init\": str(base.ls_init),\n        \"scale\": \"2\" if is_x2 else \"1\",\n        \"latent_input\": \"x0_hat_norm\",\n    }\n    if is_x2:\n        meta[\"depth_x2\"] = str(head.depth_x2)\n    if is_v2:\n        meta.update(\n            {\n                \"extra_low\": str(head.extra_low),\n                \"extra_high\": str(head.extra_high),\n                \"depth_4px\": str(head.depth_4px),\n                \"depth_2px\": str(head.depth_2px),\n                \"depth_1px\": str(head.depth_1px),\n                \"c_4px\": str(head.c_4px),\n                \"c_2px\": str(head.c_2px),\n                \"c_1px\": str(head.c_1px),\n            }\n        )\n    if extra_meta:\n        for k, v in extra_meta.items():\n            meta[str(k)] = str(v)\n\n    fp32_prefixes = (\"out_proj.\", \"base.out_proj.\", \"out2_proj.\", \"detail_out.\", \"detail_norm.\")\n    sd = {}\n    for k, v in head.state_dict().items():\n        t = v.detach().cpu().contiguous()\n        if save_bf16_trunk and t.is_floating_point() and not any(k.startswith(p) for p in fp32_prefixes):\n            t = t.to(torch.bfloat16)\n        elif any(k.startswith(p) for p in fp32_prefixes):\n            t = t.to(torch.float32)\n        sd[k] = t\n\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    save_file(sd, str(path), metadata=meta)\n\n\ndef load_qwen21_pixel_head(\n    path: str | Path,\n    device: str | torch.device = \"cpu\",\n    dtype: torch.dtype = torch.float32,\n) -> QwenImage21PixelHead | QwenImage21PixelHeadX2 | QwenImage21PixelHeadV2:\n    from safetensors import safe_open\n    from safetensors.torch import load_file\n\n    path = Path(path)\n    with safe_open(str(path), framework=\"pt\", device=\"cpu\") as fh:\n        meta = dict(fh.metadata() or {})\n\n    arch = meta.get(\"arch\", \"qwen_image21_pixel_head\")\n    base = QwenImage21PixelHead(\n        d_feat=int(meta.get(\"d_feat\", \"4096\")),\n        d_lat=int(meta.get(\"d_lat\", \"64\")),\n        width=int(meta.get(\"width\", \"1024\")),\n        depth_low=int(meta.get(\"depth_low\", \"2\")),\n        depth_high=int(meta.get(\"depth_high\", \"1\")),\n        out_channels=int(meta.get(\"out_channels\", \"4\")),\n        ls_init=float(meta.get(\"ls_init\", \"1.0\")),\n    )\n    if arch == \"qwen_image21_pixel_head_v2\" or \"depth_1px\" in meta:\n        model: nn.Module = QwenImage21PixelHeadV2(\n            base=base,\n            extra_low=int(meta.get(\"extra_low\", \"1\")),\n            extra_high=int(meta.get(\"extra_high\", \"2\")),\n            depth_4px=int(meta.get(\"depth_4px\", \"2\")),\n            depth_2px=int(meta.get(\"depth_2px\", \"2\")),\n            depth_1px=int(meta.get(\"depth_1px\", \"2\")),\n            c_4px=int(meta.get(\"c_4px\", \"256\")),\n            c_2px=int(meta.get(\"c_2px\", \"128\")),\n            c_1px=int(meta.get(\"c_1px\", \"64\")),\n            ls_init=float(meta.get(\"ls_init\", \"1.0\")),\n        )\n    elif arch == \"qwen_image21_pixel_head_x2\" or \"depth_x2\" in meta:\n        model = QwenImage21PixelHeadX2(\n            base=base,\n            depth_x2=int(meta.get(\"depth_x2\", \"2\")),\n            ls_init=float(meta.get(\"ls_init\", \"1.0\")),\n        )\n    else:\n        model = base\n\n    sd = load_file(str(path), device=\"cpu\")\n    model.load_state_dict(sd, strict=True)\n    model = model.to(device=device, dtype=dtype)\n    if isinstance(model, QwenImage21PixelHeadV2):\n        model.base.out_proj.float()\n        model.detail_norm.float()\n        model.detail_out.float()\n    elif isinstance(model, QwenImage21PixelHeadX2):\n        model.base.out_proj.float()\n        model.out2_proj.float()\n    else:\n        model.out_proj.float()\n    model.meta = meta\n    model.eval()\n    return model\n",
  "tap.py": "\"\"\"Non-invasive ComfyUI ModelPatcher tap for Qwen-Image 2.1 (`QwenImage21Transformer2DModel`).\n\nLicense: MIT\nCaptures the final denoising step's pre-`proj_out` normalized hidden states `f` (`[B, H*W, 4096]`)\nand the exact CFG-combined denoised normalized latent `x0_hat_norm` (`[B, 64, H_lat, W_lat]`), supporting:\n  - CFG = 1.0 (8-step Turbo / distilled) AND CFG > 1.0 (`f_u + cfg * (f_c - f_u)`)\n  - `QwenImage21Cache` KV prefix caching ON or OFF\n  - Both Text-to-Image (T2I) and Multi-Image Reference Editing (`<image1>`, `<image2>`, ...)\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport comfy.patcher_extension\nimport torch\n\n\nTAP_WRAPPER_KEY = \"qwen21_pixel_head_tap\"\n\n\n@dataclass\nclass StepRecord:\n    sigma: float\n    forwards: list[dict] = field(default_factory=list)\n    cond_scale: Optional[float] = None\n    denoised: Optional[torch.Tensor] = None\n\n\nclass Qwen21TapState:\n    \"\"\"Stores only the latest actual DiT forward step's features on CPU.\"\"\"\n\n    def __init__(self):\n        self.current_step: Optional[StepRecord] = None\n        self.last_valid_step: Optional[StepRecord] = None\n\n    def open_step(self, sigma: float) -> None:\n        self.current_step = StepRecord(sigma=sigma)\n\n    def close_step(self) -> None:\n        if self.current_step is not None and len(self.current_step.forwards) > 0:\n            self.last_valid_step = self.current_step\n        self.current_step = None\n\n    def resolve_final_features(self) -> tuple[torch.Tensor, torch.Tensor, float]:\n        \"\"\"Return `(f_combined [B, H*W, 4096], x0_hat_norm [B, 64, H_lat, W_lat], sigma)`.\"\"\"\n        step = self.last_valid_step\n        if step is None or not step.forwards:\n            raise RuntimeError(\n                \"QwenImage21PixelDecode: No DiT features captured! Connect `Qwen-Image 2.1 Pixel Head Tap` \"\n                \"on the MODEL wire before KSampler. / 未捕获到前向特征！请将 `Qwen-Image 2.1 Pixel Head Tap` 串联在进入采样器的 MODEL 连线上。\"\n            )\n\n        # Group recorded chunks by ComfyUI's cond_or_uncond tag (0 = positive/cond, 1 = negative/uncond)\n        by_tag: dict[int, list[tuple[torch.Tensor, torch.Tensor]]] = {0: [], 1: []}\n        for rec in step.forwards:\n            cou = rec[\"cond_or_uncond\"]\n            n_chunks = len(cou)\n            f_chunks = rec[\"f\"].chunk(n_chunks, dim=0)\n            d_chunks = rec[\"denoised\"].chunk(n_chunks, dim=0)\n            for tag, fc, dc in zip(cou, f_chunks, d_chunks):\n                by_tag.setdefault(int(tag), []).append((fc, dc))\n\n        if not by_tag.get(0):\n            raise RuntimeError(\n                \"QwenImage21PixelDecode: Missing positive condition (cond=0) features at final step. / 最后一步缺少正向条件特征记录。\"\n            )\n\n        f_cond = torch.cat([p[0] for p in by_tag[0]], dim=0).float()\n        d_cond = torch.cat([p[1] for p in by_tag[0]], dim=0).float()\n\n        if by_tag.get(1) and step.cond_scale is not None and abs(step.cond_scale - 1.0) > 1e-5:\n            f_uncond = torch.cat([p[0] for p in by_tag[1]], dim=0).float()\n            d_uncond = torch.cat([p[1] for p in by_tag[1]], dim=0).float()\n            scale = float(step.cond_scale)\n            f_final = f_uncond + scale * (f_cond - f_uncond)\n            x0_hat_fallback = d_uncond + scale * (d_cond - d_uncond)\n        else:\n            f_final = f_cond\n            x0_hat_fallback = d_cond\n\n        x0_hat_norm = step.denoised.float() if step.denoised is not None else x0_hat_fallback\n        return f_final, x0_hat_norm, float(step.sigma)\n\n\ndef attach_qwen21_tap(model_patcher) -> tuple[object, Qwen21TapState]:\n    \"\"\"Clone `model_patcher` and install lightweight step + `proj_out` pre-hooks.\"\"\"\n    cloned = model_patcher.clone()\n    state = Qwen21TapState()\n\n    # Remove any prior tap hooks if re-attached\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.CALC_COND_BATCH, TAP_WRAPPER_KEY)\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.APPLY_MODEL, TAP_WRAPPER_KEY)\n\n    def calc_cond_batch_wrapper(executor, model, conds, x_in, timestep, model_options):\n        sig = float(timestep.flatten()[0].detach().cpu().item())\n        state.open_step(sig)\n        try:\n            return executor(model, conds, x_in, timestep, model_options)\n        finally:\n            state.close_step()\n\n    def apply_model_wrapper(executor, x, t, c_concat=None, c_crossattn=None, control=None, transformer_options={}, **kwargs):\n        step = state.current_step\n        cou = (transformer_options or {}).get(\"cond_or_uncond\")\n        if step is None or not cou:\n            return executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n\n        base_model = executor.class_obj\n        diff_model = getattr(base_model, \"diffusion_model\", None)\n        proj_out = getattr(diff_model, \"proj_out\", None)\n        if proj_out is None:\n            raise RuntimeError(\n                f\"QwenImage21PixelHeadTap: Model ({type(diff_model).__name__}) has no `proj_out`; expected Qwen-Image 2.1. / \"\n                f\"当前模型 ({type(diff_model).__name__}) 不包含 `proj_out`，请确认底模为 Qwen-Image 2.1。\"\n            )\n\n        rec = {\n            \"denoised\": None,\n            \"f\": None,\n            \"cond_or_uncond\": list(cou),\n        }\n\n        def _pre_proj_hook(_mod, inputs):\n            if rec[\"f\"] is None and len(inputs) > 0:\n                rec[\"f\"] = inputs[0].detach().to(device=\"cpu\", dtype=torch.bfloat16)\n\n        handle = proj_out.register_forward_pre_hook(_pre_proj_hook)\n        try:\n            # BaseModel.apply_model returns calculate_denoised(sigma, model_output, x) = x0_hat_norm\n            out = executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n        finally:\n            handle.remove()\n\n        if rec[\"f\"] is not None:\n            rec[\"denoised\"] = out.detach().to(device=\"cpu\", dtype=torch.float32)\n            step.forwards.append(rec)\n        return out\n\n    def post_cfg_hook(args):\n        # In comfy/samplers.py, `cfg_function` runs right AFTER `calc_cond_batch` returns,\n        # so `state.last_valid_step` holds the step that just finished.\n        target_step = state.current_step if state.current_step is not None else state.last_valid_step\n        if target_step is not None:\n            if \"cond_scale\" in args:\n                target_step.cond_scale = float(args[\"cond_scale\"])\n            if \"denoised\" in args and args[\"denoised\"] is not None:\n                target_step.denoised = args[\"denoised\"].detach().to(device=\"cpu\", dtype=torch.float32)\n        return args[\"denoised\"]\n\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.CALC_COND_BATCH,\n        TAP_WRAPPER_KEY,\n        calc_cond_batch_wrapper,\n    )\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.APPLY_MODEL,\n        TAP_WRAPPER_KEY,\n        apply_model_wrapper,\n    )\n    cloned.set_model_sampler_post_cfg_function(post_cfg_hook)\n    cloned.model_options[TAP_WRAPPER_KEY] = state\n    return cloned, state\n",
  "nodes.py": "\"\"\"ComfyUI Nodes for Qwen-Image 2.1 Direct Pixel Head (Zero-VAE RGB & RGBA Decoding).\n\nLicense: MIT\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nimport time\n\nimport comfy.model_management\nimport folder_paths\nimport torch\n\nfrom .qwen21_pixel_head import (\n    QwenImage21PixelHeadV2,\n    QwenImage21PixelHeadX2,\n    load_qwen21_pixel_head,\n)\nfrom .tap import attach_qwen21_tap\n\nlogger = logging.getLogger(__name__)\n\n# Register dedicated `models/pixel_heads` directory AND allow loading from `models/vae` for convenience\nPIXEL_HEAD_DIR = os.path.join(folder_paths.models_dir, \"pixel_heads\")\nos.makedirs(PIXEL_HEAD_DIR, exist_ok=True)\nif \"pixel_heads\" not in folder_paths.folder_names_and_paths:\n    folder_paths.add_model_folder_path(\"pixel_heads\", PIXEL_HEAD_DIR)\n\n\ndef _list_candidate_files() -> list[str]:\n    seen = set()\n    items = []\n    for folder_key in (\"pixel_heads\", \"vae\"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            for fn in folder_paths.get_filename_list(folder_key):\n                if fn.endswith(\".safetensors\") and fn not in seen:\n                    seen.add(fn)\n                    items.append(fn)\n    return sorted(items) if items else [\"qwen_image_2.1_pixel_head_bf16.safetensors\", \"qwen_image_2.1_pixel_head.safetensors\"]\n\n\ndef _resolve_head_path(name: str) -> str:\n    for folder_key in (\"pixel_heads\", \"vae\"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            p = folder_paths.get_full_path(folder_key, name)\n            if p and os.path.exists(p):\n                return p\n    raise FileNotFoundError(\n        f\"Pixel Head weight `{name}` not found. Place it in `ComfyUI/models/pixel_heads/` or `ComfyUI/models/vae/`. / \"\n        f\"未找到 Pixel Head 权重 `{name}`，请放入 `ComfyUI/models/pixel_heads/` 或 `ComfyUI/models/vae/`。\"\n    )\n\n\nclass QwenImage21PixelHeadLoader:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            \"required\": {\n                \"head_name\": (_list_candidate_files(),),\n                \"precision\": ([\"bf16_trunk_fp32_out\", \"fp32\"], {\"default\": \"bf16_trunk_fp32_out\"}),\n            }\n        }\n\n    RETURN_TYPES = (\"QWEN21_PIXEL_HEAD\",)\n    RETURN_NAMES = (\"pixel_head\",)\n    FUNCTION = \"load_head\"\n    CATEGORY = \"advanced/qwen_image21\"\n    TITLE = \"Load Qwen-Image 2.1 Pixel Head\"\n    DESCRIPTION = (\n        \"Load Qwen-Image 2.1 Direct RGBA Pixel Head (~49 MB BF16 / ~97 MB FP32) from models/pixel_heads or models/vae.\\n\"\n        \"从 models/pixel_heads 或 models/vae 加载 Qwen-Image 2.1 免 VAE 直出像素头权重。\"\n    )\n\n    def load_head(self, head_name: str, precision: str):\n        path = _resolve_head_path(head_name)\n        dtype = torch.bfloat16 if precision == \"bf16_trunk_fp32_out\" else torch.float32\n        head = load_qwen21_pixel_head(path, device=\"cpu\", dtype=dtype)\n        return (head,)\n\n\nclass QwenImage21PixelHeadTap:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            \"required\": {\n                \"model\": (\"MODEL\",),\n            }\n        }\n\n    RETURN_TYPES = (\"MODEL\", \"QWEN21_TAP\")\n    RETURN_NAMES = (\"model\", \"pixel_tap\")\n    FUNCTION = \"install_tap\"\n    CATEGORY = \"advanced/qwen_image21\"\n    TITLE = \"Qwen-Image 2.1 Pixel Head Tap\"\n    DESCRIPTION = (\n        \"Non-invasive hook on Qwen-Image 2.1 MODEL right before KSampler. Supports T2I, <image1> Omni Edit, Cache & CFG.\\n\"\n        \"串联在进入 KSampler 前的 MODEL 连线上，无侵入截获末步 4096 维特征（兼容文生图、多图编辑、KV 缓存与 CFG）。\"\n    )\n\n    def install_tap(self, model):\n        patched_model, tap_state = attach_qwen21_tap(model)\n        return (patched_model, tap_state)\n\n\nclass QwenImage21PixelDecode:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            \"required\": {\n                \"samples\": (\"LATENT\",),\n                \"pixel_head\": (\"QWEN21_PIXEL_HEAD\",),\n                \"pixel_tap\": (\"QWEN21_TAP\",),\n                \"output_scale\": ([\"auto\", \"1x\", \"2x\"], {\"default\": \"auto\"}),\n            }\n        }\n\n    RETURN_TYPES = (\"IMAGE\", \"MASK\", \"IMAGE\")\n    RETURN_NAMES = (\"image_rgb\", \"alpha_mask\", \"image_rgba\")\n    FUNCTION = \"decode\"\n    CATEGORY = \"advanced/qwen_image21\"\n    TITLE = \"Qwen-Image 2.1 Pixel Decode (Zero VAE)\"\n    DESCRIPTION = (\n        \"Replaces VAEDecode: decodes Qwen-Image 2.1 features directly to RGB + Alpha in ~5 ms with zero VAE VRAM.\\n\"\n        \"替代 VAEDecode：无需加载 644MB VAE，约 5 毫秒极速直出高清 RGB、Alpha 蒙版与 4 通道 RGBA 图像。\"\n    )\n\n    @torch.no_grad()\n    def decode(self, samples, pixel_head, pixel_tap, output_scale: str = \"auto\"):\n        # `samples` dependency guarantees ComfyUI executes KSampler BEFORE this decode node!\n        _ = samples[\"samples\"]\n        f_cpu, x0_hat_cpu, sigma = pixel_tap.resolve_final_features()\n\n        device = comfy.model_management.get_torch_device()\n        is_v2 = isinstance(pixel_head, QwenImage21PixelHeadV2)\n        is_x2 = isinstance(pixel_head, QwenImage21PixelHeadX2)\n        if output_scale == \"auto\":\n            scale = 2 if is_x2 else 1\n        elif output_scale == \"2x\":\n            if not is_x2:\n                raise ValueError(\n                    \"Loaded Pixel Head is 1x; load an `_x2` checkpoint for 2x output. / 当前为 1x 像素头权重，2x 直出请加载 `_x2` 权重。\"\n                )\n            scale = 2\n        else:\n            scale = 1\n\n        head_dev = pixel_head.to(device)\n        trunk_dtype = head_dev.base.feat_proj.weight.dtype if (is_x2 or is_v2) else head_dev.feat_proj.weight.dtype\n        f_dev = f_cpu.to(device=device, dtype=trunk_dtype)\n        x_dev = x0_hat_cpu.to(device=device, dtype=trunk_dtype)\n\n        if torch.cuda.is_available() and device.type == \"cuda\":\n            torch.cuda.synchronize(device)\n        t0 = time.perf_counter()\n\n        if is_v2:\n            out_nchw = head_dev(f_dev, x_dev, sigma=sigma)\n        elif is_x2:\n            out_nchw = head_dev(f_dev, x_dev, scale=scale)\n        else:\n            out_nchw = head_dev(f_dev, x_dev)\n\n        if torch.cuda.is_available() and device.type == \"cuda\":\n            torch.cuda.synchronize(device)\n        dt_ms = (time.perf_counter() - t0) * 1000.0\n\n        # Convert [-1, 1] NCHW -> [0, 1] NHWC ComfyUI IMAGE format\n        out_nhwc = ((out_nchw.float() + 1.0) * 0.5).clamp_(0.0, 1.0).movedim(1, -1).cpu()\n        # Return head to CPU so idle VRAM stays zero\n        pixel_head.to(\"cpu\")\n\n        if out_nhwc.shape[-1] == 4:\n            rgb = out_nhwc[..., :3].contiguous()\n            alpha = out_nhwc[..., 3].contiguous()\n            rgba = out_nhwc.contiguous()\n        else:\n            rgb = out_nhwc[..., :3].contiguous()\n            alpha = torch.ones(rgb.shape[:-1], dtype=torch.float32)\n            rgba = torch.cat([rgb, alpha.unsqueeze(-1)], dim=-1)\n\n        logger.info(\n            f\"[Qwen21PixelDecode] Decoded {tuple(rgb.shape[1:3])} (scale={scale}x, last_sigma={sigma:.4f}) in {dt_ms:.2f} ms (Zero VAE)\"\n        )\n        return (rgb, alpha, rgba)\n\n\nNODE_CLASS_MAPPINGS = {\n    \"QwenImage21PixelHeadLoader\": QwenImage21PixelHeadLoader,\n    \"QwenImage21PixelHeadTap\": QwenImage21PixelHeadTap,\n    \"QwenImage21PixelDecode\": QwenImage21PixelDecode,\n}\n\nNODE_DISPLAY_NAME_MAPPINGS = {\n    \"QwenImage21PixelHeadLoader\": \"Load Qwen-Image 2.1 Pixel Head\",\n    \"QwenImage21PixelHeadTap\": \"Qwen-Image 2.1 Pixel Head Tap\",\n    \"QwenImage21PixelDecode\": \"Qwen-Image 2.1 Pixel Decode (Zero VAE)\",\n}\n",
  "__init__.py": "from .nodes import NODE_CLASS_MAPPINGS, NODE_DISPLAY_NAME_MAPPINGS\n\n__all__ = [\"NODE_CLASS_MAPPINGS\", \"NODE_DISPLAY_NAME_MAPPINGS\"]\n"
}

for fname, content in FILES.items():
    (LOCAL_SRC / fname).write_text(content, encoding="utf-8")
    (CUSTOM_NODE_DRIVE / fname).write_text(content, encoding="utf-8")

print("✅ ComfyUI-QwenImage21-PixelHead (v0.1 + v0.2 双架构兼容版) 已同步至本地与 Google Drive！")

In [ ]:
# @title 2. 预热模型至 A100 本地 NVMe SSD 并安装中英小字字体库
import json
import os
import shutil
import struct
import subprocess
import sys
import time
from pathlib import Path

t0 = time.time()
DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
COMFY_DIR = Path("/content/ComfyUI")
LOCAL_MODELS = Path("/content/local_models")
LOCAL_MODELS.mkdir(parents=True, exist_ok=True)

DIT_BASE_MODE = "uncensored_omni_int8"  # @param ["uncensored_omni_int8", "official_base_int8"]

bundle = DRIVE_HUB / "ComfyUI_qwenimage21_bundle.tar.gz"
if not COMFY_DIR.exists():
    if bundle.exists():
        print(f"🚀 从 Drive 解压 ComfyUI 预置包: {bundle.name} ...")
        subprocess.run(["tar", "-xzf", str(bundle), "-C", "/content/"], check=True)
    else:
        print("📥 克隆官方最新 ComfyUI...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)], check=True)

if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

print("⚡ 安装依赖 (lpips, pyarrow, safetensors) 与中英文字体 (fonts-noto-cjk)...")
subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=False)
subprocess.run(["uv", "pip", "install", "--system", "-q", "lpips", "pyarrow", "safetensors", "huggingface_hub"], check=True)
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "aria2", "fonts-noto-cjk", "fonts-dejavu-core"], check=False)


def is_valid_safetensors(p: Path, min_mb: float = 5.0) -> bool:
    if not p.exists() or p.stat().st_size < min_mb * 1024 * 1024:
        return False
    try:
        with open(p, "rb") as f:
            n = struct.unpack("<Q", f.read(8))[0]
            if n <= 0 or n > 100 * 1024 * 1024:
                return False
            hdr = json.loads(f.read(n))
        max_end = max((v["data_offsets"][1] for k, v in hdr.items() if k != "__metadata__"), default=0)
        return (8 + n + max_end) == p.stat().st_size
    except Exception:
        return False


def stage_model(rel_sub: str, fname: str, hf_url: str, min_mb: float, save_to_drive: bool = True) -> Path:
    local_p = LOCAL_MODELS / rel_sub / fname
    local_p.parent.mkdir(parents=True, exist_ok=True)
    if is_valid_safetensors(local_p, min_mb):
        print(f"  ✅ NVMe 已就绪: {fname} ({local_p.stat().st_size/1e9:.2f} GB)")
        return local_p

    drive_p = DRIVE_HUB / "models" / rel_sub / fname
    drive_p.parent.mkdir(parents=True, exist_ok=True)
    if is_valid_safetensors(drive_p, min_mb):
        print(f"  ⚡ 从 Drive 预热至本地 NVMe: {fname} ...")
        shutil.copyfile(drive_p, local_p)
        return local_p

    print(f"  ⬇️ 本地 NVMe 高速直拉: {fname} ...")
    tmp_dl = local_p.with_suffix(".dl")
    subprocess.run([
        "aria2c", "--console-log-level=warn", "--file-allocation=none",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        hf_url, "-d", str(local_p.parent), "-o", tmp_dl.name,
    ], check=True)
    os.replace(tmp_dl, local_p)
    assert is_valid_safetensors(local_p, min_mb), f"下载文件头校验失败: {local_p}"
    if save_to_drive:
        shutil.copyfile(local_p, drive_p)
    return local_p


if DIT_BASE_MODE == "uncensored_omni_int8":
    DIT_PATH = stage_model(
        "diffusion_models",
        "qwen-image-2.1-step90000-INT8-ConvRot-comfy.safetensors",
        "https://huggingface.co/chatrlabs/Qwen-Image-2.1-Uncensored-Omni-Quantized/resolve/main/qwen-image-2.1-step90000-INT8-ConvRot-comfy.safetensors",
        min_mb=5500,
    )
else:
    DIT_PATH = stage_model(
        "diffusion_models",
        "qwen_image_2.1_int8_convrot.safetensors",
        "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        min_mb=5500,
        save_to_drive=False,
    )

TE_PATH = stage_model(
    "text_encoders",
    "qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors",
    "https://huggingface.co/DreamFast/Qwen3-VL-8B-Heretic-1.3.0/resolve/main/comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors",
    min_mb=7500,
)

VAE_PATH = stage_model(
    "vae",
    "qwen_image_2.1_vae_bf16.safetensors",
    "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
    min_mb=450,
)

LORA_8STEP_PATH = stage_model(
    "loras",
    "p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors",
    "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/loras/p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors",
    min_mb=200,
)

V1_HEAD_PATH = stage_model(
    "pixel_heads",
    "qwen_image_2.1_pixel_head.safetensors",
    "https://huggingface.co/pocis/Qwen-Image-2.1-PixelHead/resolve/main/qwen_image_2.1_pixel_head.safetensors",
    min_mb=85,
)

for p in (str(COMFY_DIR), "/content"):
    if p not in sys.path:
        sys.path.insert(0, p)

import folder_paths
for k, sub in [
    ("diffusion_models", "diffusion_models"),
    ("text_encoders", "text_encoders"),
    ("vae", "vae"),
    ("loras", "loras"),
]:
    folder_paths.add_model_folder_path(k, str(LOCAL_MODELS / sub))

print(f"\n🎉 环境与 NVMe 模型预热全部完成！耗时: {time.time() - t0:.1f}s")

In [ ]:
# @title 3. 构建 1024×1024 高频蒸馏数据池（真机多步 CFG=3.5/Turbo 人像与小字轨迹 + 高清原图缓存）
import gc
import io
import json
import math
import random
import sys
import time
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image, ImageDraw, ImageFont

import comfy.model_management
import comfy.sample
import comfy.samplers
import comfy.sd
import comfy.utils

if "/content/ComfyUI-QwenImage21-PixelHead" not in sys.path:
    sys.path.insert(0, "/content")
from importlib import import_module
tap_mod = import_module("ComfyUI-QwenImage21-PixelHead.tap")
attach_qwen21_tap = tap_mod.attach_qwen21_tap

CACHE_V2 = Path("/content/qwen21_v02_cache")
COND_DIR = CACHE_V2 / "conds"
TRAJ_DIR = CACHE_V2 / "onpolicy_1024"
CLEAN_IMG_DIR = CACHE_V2 / "clean_imgs_1024"
CLEAN_LAT_DIR = CACHE_V2 / "clean_latents_1024"
for d in (COND_DIR, TRAJ_DIR, CLEAN_IMG_DIR, CLEAN_LAT_DIR):
    d.mkdir(parents=True, exist_ok=True)

RES = 1024
ONPOLICY_COUNT = 160    # @param {type:"integer"}  # 128 train + 32 held-out test (real KSampler 8-step & 20-step CFG=3.5!)
CLEAN_SUPP_COUNT = 440  # @param {type:"integer"}  # High-res >=700px real/art + CJK/EN small-text cards + RGBA cutouts

# -------------------------------------------------------------------------
# 3.1 专项高频提示词库（近景人像毛孔睫毛 + 中英双语极小字车票/菜单/路牌 + 细线插画）
# -------------------------------------------------------------------------
PROMPT_BANK = [
    # --- A. Portrait Close-Up & Skin/Eyelash Micro-Detail (26 prompts) ---
    ("portrait", "Extreme close-up studio portrait of a young East Asian woman, natural skin pores, fine individual eyelashes, sharp iris catchlights, stray hair strands in warm rim light, 85mm f/1.8 lens, uncompressed photo."),
    ("portrait", "Close-up portrait of an elderly craftsman with weathered skin wrinkles, silver beard stubble, clear spectacles reflecting workshop window light, crisp skin texture, documentary photography."),
    ("portrait", "Medium close-up portrait of a teenage boy in a white tech jacket by a sunny window, fluffy white hair strands, sharp golden-brown eyes, subtle freckles, clean natural skin texture."),
    ("portrait", "Editorial beauty close-up of a woman with dewy skin, fine eyebrow hairs, crisp lip texture, delicate gold earrings, soft studio key light, razor-sharp focus on eyes."),
    ("portrait", "Candid street portrait of a barista smiling in morning sunlight, sharp facial hair, detailed denim apron weave, natural skin micro-texture, shallow depth of field."),
    ("portrait", "Close-up portrait of a young girl holding a transparent glass prism, rainbow light refraction across cheek, crisp eyelashes, fine hair wisps, 85mm prime lens."),
    ("portrait", "Dramatic low-key Rembrandt lighting portrait of a violinist, sharp eye highlights, skin pores, fine wood grain on violin and horsehair bow strands."),
    ("portrait", "Studio headshot of a glasses-wearing software engineer, crisp reflection on anti-reflective lenses, detailed knit sweater fibres and realistic skin texture."),
    ("portrait", "Close-up portrait of a traditional bride in intricate embroidered red silk collar, fine metallic threadwork, sharp hairline and realistic skin pores."),
    ("portrait", "Outdoor golden-hour portrait of a hiker with windblown hair strands, subtle sun-kissed freckles, crisp nylon jacket ripstop grid texture."),
    ("portrait", "Monochrome silver-gelatin fine-art portrait close-up, high micro-contrast skin texture, sharp iris radial patterns, crisp catchlights."),
    ("portrait", "Half-body portrait of a young woman in a lace-trimmed Renaissance collar, fine fabric pleats, realistic skin tone and sharp facial features."),
    ("portrait", "Close-up of a child blowing dandelion seeds, sharp focus on eyes, eyelashes, and delicate white pappus filaments floating in air."),
    # --- B. Small Typography & Bilingual Tickets / Signs / UI (24 prompts, including exact Issue #1 ticket!) ---
    ("small_text", "高质量日本 2D 二次元插画，赛璐璐平涂上色与锐利手绘黑色线稿。一位少年坐在阳光洒入的列车窗边，手拿印有清晰字样 \"QWEN 2.1\" 与座位号 \"CAR 04 SEAT 12A\" 的车票，衣褶分明，小字笔画锐利清晰。"),
    ("small_text", "Close-up photograph of a hand holding a printed commuter train ticket reading \"QWEN-IMAGE 2.1 EXPRESS · 2026-11-23 · PLATFORM 07 · FARE ¥48.50\", crisp small thermal-printed characters and barcode."),
    ("small_text", "Overhead flat-lay photo of an open bilingual specialty coffee menu card on a wooden table, crisp small Chinese and English item names and prices like \"手冲埃塞俄比亚 Pour Over ¥38\", sharp small serif and sans-serif text."),
    ("small_text", "Street photography of a Tokyo railway station information board with crisp multi-line small Japanese, Chinese, and English directional text, platform numbers, and timetable digits."),
    ("small_text", "Macro product shot of a mechanical camera lens barrel showing finely engraved small aperture numbers \"f/1.4  2  2.8  4  5.6  8  11  16\" and focal distance scale in crisp white and orange paint."),
    ("small_text", "Clean graphic design of a tech conference attendee badge on a lanyard reading \"FASTFRAMES 2026 · SPEAKER · PIXELHEAD V0.2 · HALL B-104\", sharp small subtitle text and QR grid."),
    ("small_text", " Vintage apothecary glass bottle label with crisp small botanical Latin typography \" TINCTURA LAVANDULAE · 125 ML · BATCH NO. 0492\", sharp fine ink border and tiny lettering."),
    ("small_text", "2D anime bookstore scene where a girl holds a hardcover book titled \"DIRECT PIXEL READOUT\" with a small subtitle \"Zero-VAE Fast Inference\", crisp readable small letters on spine and cover."),
    ("small_text", "Close-up of a retro newspaper front page with a bold headline \"PIXEL HEAD V2\" and crisp multi-column small subheadings, sharp ink print on textured newsprint."),
    ("small_text", "Clean UI dashboard mockup card displaying model benchmark metrics \"Latency: 9.2 ms | PSNR: 36.4 dB | VRAM: 0.2 GB\" in crisp 12pt sans-serif typography."),
    ("small_text", " Cyberpunk alley neon and enamel warning sign in Chinese and English reading \"高压危险 HIGH VOLTAGE · 严禁攀爬 AUTHORIZED PERSONNEL ONLY 24V\", sharp small stencil letters."),
    ("small_text", "Close-up of a boarding pass and passport cover on an airport desk reading \"FLIGHT QF210 · GATE B18 · BOARDING 14:35 · SEAT 22K\", razor-sharp small printed characters."),
    # --- C. Fine Lineart, Etching, Dark Velvet & Anime (14 prompts) ---
    ("fine_art", "Intricate copperplate etching illustration of an astronomical clockwork mechanism, dense cross-hatching lines, crisp 1px black ink strokes on cream paper."),
    ("fine_art", "Traditional Chinese ink wash and fine-line gongbi painting of white plum blossoms on textured xuan paper, sharp brush tip hairlines and subtle ink gradations."),
    ("fine_art", "Crisp 2D cel-shaded anime illustration of a cyber-dragon boy with fluffy white hair and jade horns wearing a white technical jacket with glowing cyan seams, razor-sharp ink outlines."),
    ("fine_art", "Low-key still life oil painting of dark emerald velvet drapery, antique brass compass, and white lace handkerchief, smooth shadow transitions without banding and crisp lace threads."),
    ("fine_art", "Detailed architectural pen-and-ink drawing of a Gothic cathedral facade, ultra-fine stone tracery lines and sharp geometric window mullions."),
    ("fine_art", "High-resolution macro wildlife photo of an owl feather plumage and sharp amber eyes, crisp individual barbules and fine texture."),
    ("fine_art", "Woodcut print of mountain waves and pine needles, dense parallel carved lines without checkerboard artifacts, sharp black-and-white contrast."),
]

PROMPT_MANIFEST = CACHE_V2 / "prompts_manifest.json"
if not (COND_DIR / "uncond.pt").exists() or not PROMPT_MANIFEST.exists():
    print(f"🧠 [3.1] 使用 Qwen3-VL-8B 编码 {len(PROMPT_BANK)} 条高频人像/小字/线稿提示词与负向条件...")
    clip = comfy.sd.load_clip(
        ckpt_paths=[str(TE_PATH)],
        embedding_directory=None,
        clip_type=comfy.sd.CLIPType.QWEN_IMAGE,
    )
    tok_u = clip.tokenize("", images=[], keep_vision=True, prevent_empty_text=True)
    cond_u = clip.encode_from_tokens_scheduled(tok_u)
    torch.save(cond_u, COND_DIR / "uncond.pt")

    p_records = []
    for idx_p, (dom, ptxt) in enumerate(PROMPT_BANK):
        tok_c = clip.tokenize(ptxt, images=[], keep_vision=True, prevent_empty_text=True)
        cond_c = clip.encode_from_tokens_scheduled(tok_c)
        cfn = f"cond_{idx_p:03d}.pt"
        torch.save(cond_c, COND_DIR / cfn)
        p_records.append({"idx": idx_p, "domain": dom, "prompt": ptxt, "cond_file": cfn})

    PROMPT_MANIFEST.write_text(json.dumps(p_records, ensure_ascii=False, indent=2), encoding="utf-8")
    del clip
    comfy.model_management.unload_all_models()
    gc.collect()
    torch.cuda.empty_cache()
    print("  ✅ 文本编码完成，已彻底卸载 10GB 文本编码器！")
else:
    p_records = json.loads(PROMPT_MANIFEST.read_text(encoding="utf-8"))
    print(f"⏭️ 检测到已编码的 {len(p_records)} 条提示词缓存，直接复用。")

# -------------------------------------------------------------------------
# 3.2 真机 1024×1024 多步采样轨迹生成（50% 8-step Turbo CFG=1.0 + 50% 20-step Base CFG=3.5）
# -------------------------------------------------------------------------
TRAJ_MANIFEST = CACHE_V2 / "traj_manifest.json"
existing_traj = json.loads(TRAJ_MANIFEST.read_text(encoding="utf-8")) if TRAJ_MANIFEST.exists() else []
done_ids = {r["id"] for r in existing_traj if (TRAJ_DIR / r["file"]).exists()}

if len(done_ids) < ONPOLICY_COUNT:
    print(f"🎨 [3.2] 在 A100 上生成真机 1024×1024 多步采样蒸馏轨迹 ({len(done_ids)}/{ONPOLICY_COUNT})...")
    model_base = comfy.sd.load_diffusion_model(str(DIT_PATH))
    lora_sd = comfy.utils.load_torch_file(str(LORA_8STEP_PATH), safe_load=True)
    model_turbo, _ = comfy.sd.load_lora_for_models(model_base, None, lora_sd, 1.0, 0.0)
    del lora_sd

    vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))
    uncond_obj = torch.load(COND_DIR / "uncond.pt", map_location="cpu", weights_only=False)
    cond_objs = [torch.load(COND_DIR / r["cond_file"], map_location="cpu", weights_only=False) for r in p_records]

    traj_list = [r for r in existing_traj if r["id"] in done_ids]
    t_g0 = time.time()

    for tid in range(ONPOLICY_COUNT):
        if tid in done_ids:
            continue
        pr = p_records[tid % len(p_records)]
        # Group in blocks of 8 to avoid LoRA hot-swap overhead while keeping exact 50/50 split across train & test!
        use_turbo = ((tid // 4) % 2 == 0)
        steps = 8 if use_turbo else 20
        cfg = 1.0 if use_turbo else 3.5
        seed = 20262100 + tid

        base_patcher = model_turbo if use_turbo else model_base
        tapped_patcher, tap_state = attach_qwen21_tap(base_patcher)

        noise = comfy.sample.prepare_noise(torch.zeros([1, 64, RES // 16, RES // 16]), seed)
        latent_image = torch.zeros([1, 64, RES // 16, RES // 16])

        with torch.no_grad():
            samples_raw = comfy.sample.sample(
                tapped_patcher,
                noise,
                steps,
                cfg,
                "euler",
                "simple",
                cond_objs[pr["idx"]],
                uncond_obj,
                latent_image,
                denoise=1.0,
                disable_noise=False,
                start_step=None,
                last_step=None,
                force_full_denoise=True,
                noise_mask=None,
                callback=None,
                disable_pbar=True,
                seed=seed,
            )
            f_final, x0_hat_norm, last_sigma = tap_state.resolve_final_features()
            rgba_01 = vae.decode(samples_raw.cpu())
            if rgba_01.shape[-1] == 3:
                rgba_01 = torch.cat([rgba_01, torch.ones((*rgba_01.shape[:3], 1), dtype=rgba_01.dtype)], dim=-1)
            tgt_nchw = (rgba_01[0].permute(2, 0, 1).float() * 2.0 - 1.0).clamp_(-1.0, 1.0).to(torch.bfloat16).cpu()

        gh, gw = x0_hat_norm.shape[-2], x0_hat_norm.shape[-1]
        fn_pt = f"traj_{tid:04d}.pt"
        torch.save(
            {
                "f": f_final[0].view(gh, gw, -1).to(torch.bfloat16).cpu(),
                "x": x0_hat_norm[0].to(torch.bfloat16).cpu(),
                "sigma": float(last_sigma),
                "tgt": tgt_nchw,
            },
            TRAJ_DIR / fn_pt,
        )
        split = "test" if tid >= (ONPOLICY_COUNT - 32) else "train"
        traj_list.append(
            {
                "id": tid,
                "file": fn_pt,
                "domain": pr["domain"],
                "mode": "turbo_8s" if use_turbo else "base_20s_cfg35",
                "sigma": float(last_sigma),
                "split": split,
                "prompt": pr["prompt"],
            }
        )
        if (tid + 1) % 16 == 0 or (tid + 1) == ONPOLICY_COUNT:
            TRAJ_MANIFEST.write_text(json.dumps(traj_list, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"  [On-Policy 1024] {tid + 1}/{ONPOLICY_COUNT} ({pr['domain']}, {'Turbo' if use_turbo else 'CFG3.5'}, sig={last_sigma:.4f}) elapsed={time.time()-t_g0:.1f}s")

    TRAJ_MANIFEST.write_text(json.dumps(traj_list, ensure_ascii=False, indent=2), encoding="utf-8")
    del model_base, model_turbo, vae
    comfy.model_management.unload_all_models()
    gc.collect()
    torch.cuda.empty_cache()
else:
    traj_list = existing_traj
    print(f"⏭️ 检测到已完成的 {len(traj_list)} 条 1024×1024 真机多步轨迹缓存，直接复用！")

# -------------------------------------------------------------------------
# 3.3 补充高分辨率 (>=700px) 原图 + 中英双语 9px~28px 小字排版卡 + 原生 RGBA 贴纸缓存
# -------------------------------------------------------------------------
CLEAN_MANIFEST = CACHE_V2 / "clean_manifest.json"


def load_cjk_or_sans_font(sz: int):
    for fp in [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    ]:
        if Path(fp).exists():
            return ImageFont.truetype(fp, sz)
    return ImageFont.load_default()


def make_small_text_sheet(idx: int, size: int = 1024) -> Image.Image:
    rng = random.Random(77000 + idx)
    palettes = [
        ((248, 246, 240, 255), (22, 25, 32, 255), (185, 38, 32, 255)),
        ((18, 24, 36, 255), (236, 242, 250, 255), (56, 189, 248, 255)),
        ((255, 255, 255, 255), (15, 23, 42, 255), (21, 128, 61, 255)),
        ((242, 236, 222, 255), (38, 28, 20, 255), (161, 98, 7, 255)),
    ]
    bg, fg, acc = palettes[idx % len(palettes)]
    im = Image.new("RGBA", (size, size), bg)
    dr = ImageDraw.Draw(im)

    # Draw realistic multi-card tickets, badges, menus & micro-labels (9px .. 32px!)
    for card_i in range(4):
        cx = (card_i % 2) * (size // 2) + 28
        cy = (card_i // 2) * (size // 2) + 28
        cw, ch = size // 2 - 56, size // 2 - 56
        dr.rounded_rectangle([cx, cy, cx + cw, cy + ch], radius=14, outline=acc, width=2)
        dr.line([(cx + 16, cy + 56), (cx + cw - 16, cy + 56)], fill=fg, width=1)

        dr.text((cx + 18, cy + 16), f"QWEN 2.1 EXPRESS · 检票口 0{card_i+1}A", font=load_cjk_or_sans_font(22), fill=acc)
        lines = [
            (18, f"车次 G2026  座位 0{rng.randint(1,9)}车1{rng.randint(0,9)}F  票价 ¥{rng.randint(35,280)}.50"),
            (15, "FastFrames Zero-VAE Direct Pixel Readout v0.2 (A100)"),
            (13, "小字清晰度专项校准：皮肤毛孔 · 睫毛 · 车票细字 · 1024px"),
            (12, "0123456789  PSNR 36.8dB  LPIPS 0.048  f/1.4  ISO 100  1/250s"),
            (11, "abcdefghijklmnopqrstuvwxyz  ABCDEFGHIJKLMNOPQRSTUVWXYZ"),
            (10, "注意：限乘当日当次车 · 保持票面整洁 · VALID FOR DATE SHOWN ONLY"),
            (9,  "SERIAL NO. QF-2026-1123-88412-A100  |  BATCH 0492  |  CHECK OK"),
        ]
        yy = cy + 72
        for fsz, txt in lines:
            dr.text((cx + 18, yy), txt, font=load_cjk_or_sans_font(fsz), fill=fg)
            yy += fsz + 16

        # Barcode fine vertical lines (1px / 2px / 3px)
        bx = cx + 20
        while bx < cx + cw - 30:
            bw = rng.choice([1, 1, 2, 3])
            dr.rectangle([bx, cy + ch - 46, bx + bw - 1, cy + ch - 16], fill=fg)
            bx += bw + rng.choice([1, 2, 3])
    return im


def fetch_highres_parquet_images(dataset_id: str, max_count: int, domain_tag: str, start_idx: int) -> list[dict]:
    items, shard_urls = [], []
    try:
        req = urllib.request.Request(f"https://huggingface.co/api/datasets/{dataset_id}/parquet", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            pq_tree = json.loads(r.read().decode("utf-8"))
        for cfg_map in pq_tree.values():
            if isinstance(cfg_map, dict):
                for split_list in cfg_map.values():
                    if isinstance(split_list, list):
                        shard_urls.extend(split_list)
    except Exception:
        pass

    for idx_sh, sh_url in enumerate(shard_urls):
        if len(items) >= max_count:
            break
        local_pq = CACHE_V2 / f"_tmp_{dataset_id.replace('/', '_')}_{idx_sh:03d}.parquet"
        if not local_pq.exists():
            urllib.request.urlretrieve(sh_url, str(local_pq))
        table = pq.read_table(local_pq)
        img_col = next((c for c in table.column_names if "image" in c.lower()), None)
        if img_col is not None:
            for row_val in table[img_col].to_pylist():
                if len(items) >= max_count:
                    break
                raw_bytes = row_val.get("bytes") if isinstance(row_val, dict) else None
                if not raw_bytes:
                    continue
                try:
                    im = Image.open(io.BytesIO(raw_bytes))
                    # Strictly keep genuinely sharp high-res images (min side >= 640) so we never train on blurry upscales!
                    if min(im.size) < 640:
                        continue
                    im = im.convert("RGBA")
                    w, h = im.size
                    side = min(w, h)
                    im_sq = im.crop(((w - side) // 2, (h - side) // 2, (w + side) // 2, (h + side) // 2)).resize((RES, RES), Image.Resampling.LANCZOS)
                    cid = start_idx + len(items)
                    fn = f"clean_{cid:04d}_{domain_tag}.png"
                    im_sq.save(CLEAN_IMG_DIR / fn)
                    items.append({"id": cid, "image": fn, "domain": domain_tag})
                except Exception:
                    continue
        local_pq.unlink(missing_ok=True)
    return items


if not CLEAN_MANIFEST.exists():
    print(f"📦 [3.3] 构建 {CLEAN_SUPP_COUNT} 张 1024×1024 高清原图 + 双语极小字卡 + RGBA 透明校准集...")
    c_rows = []
    c_rows.extend(fetch_highres_parquet_images("huggan/wikiart", int(CLEAN_SUPP_COUNT * 0.40), "fine_art", 0))
    n_typo = int(CLEAN_SUPP_COUNT * 0.40)
    for k in range(n_typo):
        cid = len(c_rows)
        im_t = make_small_text_sheet(k, RES)
        fn = f"clean_{cid:04d}_small_text.png"
        im_t.save(CLEAN_IMG_DIR / fn)
        c_rows.append({"id": cid, "image": fn, "domain": "small_text"})

    # Native 4ch RGBA cutouts from sharp art & typography sheets
    n_rgba = max(24, CLEAN_SUPP_COUNT - len(c_rows))
    for k in range(n_rgba):
        cid = len(c_rows)
        src_im = Image.open(CLEAN_IMG_DIR / c_rows[k % len(c_rows)]["image"]).convert("RGBA")
        mask = Image.new("L", (RES, RES), 0)
        mdr = ImageDraw.Draw(mask)
        pad = 96 + (k % 5) * 24
        if k % 2 == 0:
            mdr.ellipse([pad, pad, RES - pad, RES - pad], fill=255)
        else:
            mdr.rounded_rectangle([pad, pad, RES - pad, RES - pad], radius=72, fill=255)
        src_im.putalpha(mask)
        arr = np.array(src_im)
        arr[arr[..., 3] == 0, :3] = 0
        fn = f"clean_{cid:04d}_rgba.png"
        Image.fromarray(arr, "RGBA").save(CLEAN_IMG_DIR / fn)
        c_rows.append({"id": cid, "image": fn, "domain": "fine_art"})

    vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))
    for idx_c, r in enumerate(c_rows):
        im = Image.open(CLEAN_IMG_DIR / r["image"]).convert("RGBA")
        arr = torch.from_numpy(np.array(im)).float().div_(255.0).unsqueeze(0)
        with torch.no_grad():
            z = vae.encode(arr)
        lfn = f"lat_{r['id']:04d}.pt"
        torch.save(z.to(torch.bfloat16).cpu(), CLEAN_LAT_DIR / lfn)
        r["latent"] = lfn
        r["cond_idx"] = r["id"] % len(p_records)
        if (idx_c + 1) % 100 == 0 or (idx_c + 1) == len(c_rows):
            print(f"  已编码 1024 Latent: {idx_c + 1}/{len(c_rows)}")
    del vae
    comfy.model_management.cleanup_models_gc()
    gc.collect()
    torch.cuda.empty_cache()
    CLEAN_MANIFEST.write_text(json.dumps(c_rows, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    c_rows = json.loads(CLEAN_MANIFEST.read_text(encoding="utf-8"))
    print(f"⏭️ 检测到已编码的 {len(c_rows)} 张 1024×1024 高清补充潜变量缓存，直接复用！")

print(f"\n🎉 数据池全部就绪！真机多步轨迹: {len(traj_list)} 条 (含 32 条独立盲测集) + 高清小字/艺术原图: {len(c_rows)} 张")

In [ ]:
# @title 4. Warm-Start 自 v0.1 + A100 全分辨率感知/频域锐化训练 (LPIPS-VGG + 2D FFT + EMA 0.999)
from collections import deque
import copy
import gc
import importlib
import json
import math
import random
import shutil
import time
from pathlib import Path

import lpips
import torch
import torch.nn as nn
import torch.nn.functional as F

import comfy.model_management
import comfy.sd
import comfy.utils

ph_mod = importlib.import_module("ComfyUI-QwenImage21-PixelHead.qwen21_pixel_head")
importlib.reload(ph_mod)
QwenImage21PixelHeadV2 = ph_mod.QwenImage21PixelHeadV2
load_qwen21_pixel_head = ph_mod.load_qwen21_pixel_head
save_qwen21_pixel_head = ph_mod.save_qwen21_pixel_head

# @markdown ### A100 训练超参数（默认 3600 Rollouts × 12 步 = 43,200 次全分辨率+感知锐化更新，A100 约 32~40 分钟）：
TOTAL_ROLLOUTS = 3600       # @param {type:"integer"}
UPDATES_PER_ROLLOUT = 12    # @param {type:"integer"}
REPLAY_SIZE = 96            # @param {type:"integer"}
LR_DETAIL = 3.0e-4          # @param {type:"number"}
LR_BASE = 8.0e-5            # @param {type:"number"}
EMA_DECAY = 0.999           # @param {type:"number"}
SAVE_EVERY = 400            # @param {type:"integer"}
RESUME_IF_EXISTS = True     # @param {type:"boolean"}

DEVICE = torch.device("cuda")
CACHE_V2 = Path("/content/qwen21_v02_cache")
PROJECT_V2 = Path("/content/drive/MyDrive/FastFrames_QwenImage21_v02")
CKPT_V2 = PROJECT_V2 / "checkpoints"
PIXEL_HEADS_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads")
VAE_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/vae")

traj_manifest = json.loads((CACHE_V2 / "traj_manifest.json").read_text(encoding="utf-8"))
traj_train = [r for r in traj_manifest if r["split"] == "train"]
traj_test = [r for r in traj_manifest if r["split"] == "test"]
clean_rows = json.loads((CACHE_V2 / "clean_manifest.json").read_text(encoding="utf-8"))
prompts_manifest = json.loads((CACHE_V2 / "prompts_manifest.json").read_text(encoding="utf-8"))

# 1. 加载冻结 DiT、8-Step LoRA、Teacher WanVAE 与 VGG-LPIPS 感知网络
comfy.model_management.unload_all_models()
gc.collect()
torch.cuda.empty_cache()

model_base = comfy.sd.load_diffusion_model(str(DIT_PATH))
lora_sd = comfy.utils.load_torch_file(str(LORA_8STEP_PATH), safe_load=True)
model_turbo, _ = comfy.sd.load_lora_for_models(model_base, None, lora_sd, 1.0, 0.0)
del lora_sd
teacher_vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))

lpips_vgg = lpips.LPIPS(net="vgg", verbose=False).to(DEVICE).eval()
lpips_vgg.requires_grad_(False)

cond_tensors = []
for pr in prompts_manifest:
    c_obj = torch.load(CACHE_V2 / "conds" / pr["cond_file"], map_location="cpu", weights_only=False)
    cond_tensors.append(c_obj[0][0].detach().to(torch.bfloat16))
uncond_tensor = torch.load(CACHE_V2 / "conds" / "uncond.pt", map_location="cpu", weights_only=False)[0][0].detach().to(torch.bfloat16)

# 2. Warm-Start QwenImage21PixelHeadV2 from v0.1!
v1_base = load_qwen21_pixel_head(V1_HEAD_PATH, device="cpu", dtype=torch.float32)
head = QwenImage21PixelHeadV2(base=v1_base).to(DEVICE).train()
ema_head = copy.deepcopy(head).eval().requires_grad_(False)

n_tot = sum(p.numel() for p in head.parameters())
print(f"🚀 初始化 QwenImage21PixelHeadV2 (Warm-Started from v0.1): 总参数量 = {n_tot/1e6:.2f}M")

base_params = list(head.base.parameters())
base_ids = {id(p) for p in base_params}
detail_params = [p for p in head.parameters() if id(p) not in base_ids]

optimizer = torch.optim.AdamW(
    [
        {"params": base_params, "lr": LR_BASE, "base_lr": LR_BASE},
        {"params": detail_params, "lr": LR_DETAIL, "base_lr": LR_DETAIL},
    ],
    betas=(0.9, 0.99),
    weight_decay=1e-2,
)

total_steps = TOTAL_ROLLOUTS * UPDATES_PER_ROLLOUT
warmup_steps = 250


def update_lr(step: int):
    warm = min(1.0, (step + 1) / warmup_steps)
    prog = min(1.0, step / max(1, total_steps))
    cos_f = 0.08 + 0.92 * 0.5 * (1.0 + math.cos(math.pi * prog))
    for pg in optimizer.param_groups:
        pg["lr"] = pg["base_lr"] * warm * cos_f


@torch.no_grad()
def update_ema(decay: float):
    for p_ema, p_src in zip(ema_head.parameters(), head.parameters()):
        p_ema.data.mul_(decay).add_(p_src.data, alpha=1.0 - decay)


def high_freq_sharpness_loss(pred: torch.Tensor, tgt: torch.Tensor) -> tuple[torch.Tensor, dict]:
    """Sharpness-first composite loss designed specifically for crisp small text & portrait micro-texture."""
    p_rgb, t_rgb = pred[:, :3].float(), tgt[:, :3].float()
    p_a, t_a = pred[:, 3:].float(), tgt[:, 3:].float()

    # 1. Charbonnier L1 + gentle MSE (avoids quadratic edge blurring of v0.1's 5.0 * MSE!)
    charb = torch.sqrt((p_rgb - t_rgb) ** 2 + 1e-6).mean()
    mse = ((p_rgb - t_rgb) ** 2).mean()

    # 2. Edge-weighted 1px & 2px spatial gradient loss (5x focus on crisp text strokes, eyelashes & hair!)
    dy_t = t_rgb[:, :, 1:, :] - t_rgb[:, :, :-1, :]
    dx_t = t_rgb[:, :, :, 1:] - t_rgb[:, :, :, :-1]
    dy_p = p_rgb[:, :, 1:, :] - p_rgb[:, :, :-1, :]
    dx_p = p_rgb[:, :, :, 1:] - p_rgb[:, :, :, :-1]
    w_y = 1.0 + 4.0 * dy_t.abs().mean(dim=1, keepdim=True).clamp(0.0, 1.0)
    w_x = 1.0 + 4.0 * dx_t.abs().mean(dim=1, keepdim=True).clamp(0.0, 1.0)
    grad_1px = (w_y * (dy_p - dy_t).abs()).mean() + (w_x * (dx_p - dx_t).abs()).mean()

    dy2 = ((p_rgb[:, :, 2:, :] - p_rgb[:, :, :-2, :]) - (t_rgb[:, :, 2:, :] - t_rgb[:, :, :-2, :])).abs().mean()
    dx2 = ((p_rgb[:, :, :, 2:] - p_rgb[:, :, :, :-2]) - (t_rgb[:, :, :, 2:] - t_rgb[:, :, :, :-2])).abs().mean()
    grad_loss = grad_1px + 0.5 * (dy2 + dx2)

    # 3. VGG-16 Perceptual LPIPS loss (runs fast in FP32/BF16 on A100)
    lp_loss = lpips_vgg(p_rgb.clamp(-1.0, 1.0), t_rgb.clamp(-1.0, 1.0)).mean()

    # 4. 2D Radial-Weighted Log-FFT Amplitude Loss (eliminates high-frequency rolloff)
    pf = torch.fft.rfft2(p_rgb, norm="ortho")
    tf = torch.fft.rfft2(t_rgb, norm="ortho")
    p_mag = torch.log1p(torch.abs(pf) + 1e-6)
    t_mag = torch.log1p(torch.abs(tf) + 1e-6)
    fy = torch.fft.fftfreq(p_rgb.shape[-2], device=p_rgb.device)[:, None]
    fx = torch.fft.rfftfreq(p_rgb.shape[-1], device=p_rgb.device)[None, :]
    rad_w = (1.0 + 3.0 * torch.sqrt(fy ** 2 + fx ** 2))[None, None]
    fft_loss = (rad_w * (p_mag - t_mag).abs()).mean()

    # 5. Alpha channel fidelity
    a_loss = (p_a - t_a).abs().mean() + 0.5 * ((p_a - t_a) ** 2).mean()
    a_grad = (p_a.diff(dim=-2) - t_a.diff(dim=-2)).abs().mean() + (p_a.diff(dim=-1) - t_a.diff(dim=-1)).abs().mean()

    total = (
        1.00 * charb
        + 0.40 * mse
        + 0.28 * grad_loss
        + 0.32 * lp_loss
        + 0.08 * fft_loss
        + 0.30 * a_loss
        + 0.05 * a_grad
    )
    psnr_db = -10.0 * math.log10(max(1e-10, float(mse.detach().item()) * 0.25))
    return total, {"psnr": psnr_db, "lpips": float(lp_loss.detach().item())}


@torch.no_grad()
def single_step_clean_rollout(row: dict, use_turbo: bool, sigma: float, seed: int):
    patcher = model_turbo if use_turbo else model_base
    comfy.model_management.load_models_gpu([patcher])
    bmodel = patcher.model
    lat_fmt = bmodel.latent_format

    x0_raw = torch.load(CACHE_V2 / "clean_latents_1024" / row["latent"], map_location=DEVICE, weights_only=True).float()
    x0_norm = lat_fmt.process_in(x0_raw)
    c_pos = cond_tensors[row["cond_idx"]].to(device=DEVICE, dtype=bmodel.get_dtype_inference())

    gen = torch.Generator(device=DEVICE).manual_seed(int(seed))
    xt_norm = (1.0 - sigma) * x0_norm + sigma * torch.randn(x0_norm.shape, generator=gen, device=DEVICE)
    sig_t = torch.tensor([sigma], device=DEVICE, dtype=torch.float32)

    if hasattr(bmodel.diffusion_model, "reset_prefix_cache"):
        bmodel.diffusion_model.reset_prefix_cache(False)

    cap_c, cap_u = {}, {}
    h1 = bmodel.diffusion_model.proj_out.register_forward_pre_hook(lambda _m, inp: (cap_c.setdefault("f", inp[0].detach()), None)[1])
    try:
        d_cond = bmodel.apply_model(xt_norm, sig_t, c_crossattn=c_pos).float()
    finally:
        h1.remove()

    # On 50% of Base rollouts, run actual CFG=3.5 extrapolation so clean small-text sheets also train CFG>1!
    if (not use_turbo) and (seed % 2 == 0):
        c_neg = uncond_tensor.to(device=DEVICE, dtype=bmodel.get_dtype_inference())
        if hasattr(bmodel.diffusion_model, "reset_prefix_cache"):
            bmodel.diffusion_model.reset_prefix_cache(False)
        h2 = bmodel.diffusion_model.proj_out.register_forward_pre_hook(lambda _m, inp: (cap_u.setdefault("f", inp[0].detach()), None)[1])
        try:
            d_uncond = bmodel.apply_model(xt_norm, sig_t, c_crossattn=c_neg).float()
        finally:
            h2.remove()
        cfg_s = 3.5
        f_final = cap_u["f"].float() + cfg_s * (cap_c["f"].float() - cap_u["f"].float())
        x0_hat_norm = d_uncond + cfg_s * (d_cond - d_uncond)
    else:
        f_final = cap_c["f"].float()
        x0_hat_norm = d_cond

    x0_hat_raw = lat_fmt.process_out(x0_hat_norm)
    rgba_01 = teacher_vae.decode(x0_hat_raw.cpu())
    if rgba_01.shape[-1] == 3:
        rgba_01 = torch.cat([rgba_01, torch.ones((*rgba_01.shape[:3], 1), dtype=rgba_01.dtype)], dim=-1)
    tgt_nchw = (rgba_01[0].permute(2, 0, 1).float() * 2.0 - 1.0).clamp_(-1.0, 1.0)

    gh, gw = x0_hat_norm.shape[-2], x0_hat_norm.shape[-1]
    return (
        f_final[0].view(gh, gw, -1).to(torch.bfloat16).cpu(),
        x0_hat_norm[0].to(torch.bfloat16).cpu(),
        float(sigma),
        tgt_nchw.to(torch.bfloat16).cpu(),
    )


@torch.no_grad()
def evaluate_v2(eval_model: nn.Module) -> dict:
    eval_model.eval()
    psnrs, lps = [], []
    by_dom = {}
    for r in traj_test:
        item = torch.load(CACHE_V2 / "onpolicy_1024" / r["file"], map_location="cpu", weights_only=True)
        f_in = item["f"].unsqueeze(0).to(DEVICE, torch.bfloat16)
        x_in = item["x"].unsqueeze(0).to(DEVICE, torch.bfloat16)
        tgt = item["tgt"].unsqueeze(0).to(DEVICE, torch.float32)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = eval_model(f_in, x_in, sigma=float(item["sigma"])).float()
        p01 = ((pred[:, :3] + 1.0) * 0.5).clamp(0.0, 1.0)
        t01 = ((tgt[:, :3] + 1.0) * 0.5).clamp(0.0, 1.0)
        mse_v = max(1e-10, float(((p01 - t01) ** 2).mean().item()))
        p_db = -10.0 * math.log10(mse_v)
        lp_v = float(lpips_vgg(pred[:, :3].clamp(-1, 1), tgt[:, :3].clamp(-1, 1)).mean().item())
        psnrs.append(p_db)
        lps.append(lp_v)
        by_dom.setdefault(r["domain"], []).append((p_db, lp_v))
    return {
        "psnr": sum(psnrs) / len(psnrs),
        "lpips": sum(lps) / len(lps),
        "score": (sum(psnrs) / len(psnrs)) - 25.0 * (sum(lps) / len(lps)),
        "by_dom": {k: (sum(x[0] for x in v) / len(v), sum(x[1] for x in v) / len(v)) for k, v in by_dom.items()},
    }


latest_ckpt = CKPT_V2 / "latest_v02_state.pt"
start_rollout = 0
opt_step = 0
best_score = -1e9
history = []

if RESUME_IF_EXISTS and latest_ckpt.exists():
    st = torch.load(latest_ckpt, map_location=DEVICE, weights_only=False)
    head.load_state_dict(st["head"])
    ema_head.load_state_dict(st["ema_head"])
    optimizer.load_state_dict(st["optimizer"])
    start_rollout = int(st["rollout"])
    opt_step = int(st["opt_step"])
    best_score = float(st["best_score"])
    history = list(st.get("history", []))
    print(f"🔄 从断点恢复: Rollout {start_rollout}/{TOTAL_ROLLOUTS} (best_score={best_score:.2f})")
else:
    ev0 = evaluate_v2(ema_head)
    best_score = ev0["score"]
    print(f"🔬 [Step 0 基线校验 (v0.1 Warm-Start @ 1024×1024)] PSNR={ev0['psnr']:.2f} dB | LPIPS={ev0['lpips']:.4f} | 分领域={ev0['by_dom']}")

# Preload all 128 on-policy 1024 trajectories into RAM (~6.5 GB on Colab A100 83GB RAM!) for instant zero-IO sampling
onpolicy_ram = []
for r in traj_train:
    d_pt = torch.load(CACHE_V2 / "onpolicy_1024" / r["file"], map_location="cpu", weights_only=True)
    onpolicy_ram.append((d_pt["f"], d_pt["x"], float(d_pt["sigma"]), d_pt["tgt"]))

replay = deque(maxlen=REPLAY_SIZE)
for item in onpolicy_ram[: min(48, len(onpolicy_ram))]:
    replay.append(item)

rng = random.Random(20261123 + start_rollout)
t_t0 = time.time()
run_psnr, run_lp, run_n = 0.0, 0.0, 0

for r_idx in range(start_rollout + 1, TOTAL_ROLLOUTS + 1):
    # 55% sample from cached 1024 on-policy multi-step trajectories, 45% fresh single-step rollout on clean small-text/art
    if rng.random() < 0.55:
        pair = onpolicy_ram[rng.randrange(len(onpolicy_ram))]
    else:
        crow = clean_rows[(r_idx * 131) % len(clean_rows)]
        use_t = (((r_idx - 1) // 8) % 2 == 0)
        sig = (0.2217 if (r_idx % 2 == 0) else rng.uniform(0.11, 0.30)) if use_t else rng.uniform(0.038, 0.115)
        pair = single_step_clean_rollout(crow, use_t, sig, 600000 + r_idx)
    replay.append(pair)

    head.train()
    for u_i in range(UPDATES_PER_ROLLOUT):
        update_lr(opt_step)
        bf, bx, bsig, bt = replay[rng.randrange(len(replay))]
        gh, gw = bf.shape[:2]
        # Alternate 32x32 token (512x512 px) and 44x44 token (704x704 px) crops with 2-token margin strip
        win_tok = 40 if (u_i % 4 == 0) else 28
        mar = 2
        span = min(gh, gw, win_tok + 2 * mar)
        y0 = rng.randint(0, gh - span)
        x0 = rng.randint(0, gw - span)
        mt = 0 if y0 == 0 else mar * 16
        mb = 0 if (y0 + span) == gh else mar * 16
        ml = 0 if x0 == 0 else mar * 16
        mr = 0 if (x0 + span) == gw else mar * 16

        f_s = bf[y0:y0+span, x0:x0+span].unsqueeze(0).to(DEVICE, torch.bfloat16)
        x_s = bx[:, y0:y0+span, x0:x0+span].unsqueeze(0).to(DEVICE, torch.bfloat16)
        t_s = bt[:, y0*16:(y0+span)*16, x0*16:(x0+span)*16].unsqueeze(0).to(DEVICE, torch.float32)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_s, x_s, sigma=bsig)

        h_px, w_px = pred.shape[-2], pred.shape[-1]
        p_crop = pred[..., mt:h_px-mb, ml:w_px-mr]
        t_crop = t_s[..., mt:h_px-mb, ml:w_px-mr]

        loss, m_dict = high_freq_sharpness_loss(p_crop, t_crop)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
        optimizer.step()
        update_ema(EMA_DECAY if opt_step > 100 else 0.98)

        opt_step += 1
        run_psnr += m_dict["psnr"]
        run_lp += m_dict["lpips"]
        run_n += 1

    if r_idx % 50 == 0 or r_idx == 1:
        el = time.time() - t_t0
        rate = (r_idx - start_rollout) / max(1e-3, el)
        eta_m = (TOTAL_ROLLOUTS - r_idx) / max(1e-5, rate) / 60.0
        print(
            f"[Rollout {r_idx:4d}/{TOTAL_ROLLOUTS}] "
            f"train_PSNR={run_psnr/max(1,run_n):.2f}dB | train_LPIPS={run_lp/max(1,run_n):.4f} | "
            f"VRAM={torch.cuda.max_memory_allocated()/1e9:.1f}GB | ETA={eta_m:.1f}m"
        )
        run_psnr, run_lp, run_n = 0.0, 0.0, 0

    if r_idx % SAVE_EVERY == 0 or r_idx == TOTAL_ROLLOUTS:
        ev = evaluate_v2(ema_head)
        history.append({"rollout": r_idx, **ev})
        print(
            f"  🎯 [EMA Eval @ {r_idx}] PSNR={ev['psnr']:.2f} dB | LPIPS={ev['lpips']:.4f} | "
            f"Score={ev['score']:.2f} | Domains={ev['by_dom']}"
        )
        meta_stamp = {
            "version": "0.2.0",
            "base_model": DIT_PATH.name,
            "teacher_vae": VAE_PATH.name,
            "rollouts": str(r_idx),
            "eval_psnr_db": f"{ev['psnr']:.2f}",
            "eval_lpips_vgg": f"{ev['lpips']:.4f}",
            "sigma_ranges": "0.038-0.300",
        }
        snap_bf16 = CKPT_V2 / f"qwen_image_2.1_pixel_head_v02_r{r_idx:04d}_bf16.safetensors"
        save_qwen21_pixel_head(ema_head, snap_bf16, extra_meta=meta_stamp, save_bf16_trunk=True)

        if ev["score"] >= best_score or r_idx == TOTAL_ROLLOUTS:
            best_score = max(best_score, ev["score"])
            out_fp32 = CKPT_V2 / "qwen_image_2.1_pixel_head_v02.safetensors"
            out_bf16 = CKPT_V2 / "qwen_image_2.1_pixel_head_v02_bf16.safetensors"
            save_qwen21_pixel_head(ema_head, out_fp32, extra_meta=meta_stamp, save_bf16_trunk=False)
            shutil.copyfile(snap_bf16, out_bf16)
            for dst_dir in (PIXEL_HEADS_DRIVE, VAE_DRIVE):
                shutil.copyfile(out_bf16, dst_dir / "qwen_image_2.1_pixel_head_v02_bf16.safetensors")
                shutil.copyfile(out_fp32, dst_dir / "qwen_image_2.1_pixel_head_v02.safetensors")
            print(f"  💾 已同步最优 v0.2 权重至 Drive ({out_bf16.stat().st_size/(1024*1024):.1f} MiB BF16)")

        torch.save(
            {
                "rollout": r_idx,
                "opt_step": opt_step,
                "head": head.state_dict(),
                "ema_head": ema_head.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best_score": best_score,
                "history": history,
            },
            latest_ckpt,
        )

print("\n🎉 Qwen-Image 2.1 PixelHead v0.2 A100 训练圆满完成！")

In [ ]:
# @title 5. 三方硬核同屏盲测：Official 644MB VAE vs v0.1 vs v0.2（小字车票 + 近景人像 1:1 放大检验）
import math
import time
from pathlib import Path
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image, ImageDraw

import lpips
import numpy as np
import torch
import comfy.sd
import comfy.utils

EVAL_V2 = Path("/content/drive/MyDrive/FastFrames_QwenImage21_v02/eval_report")
EVAL_V2.mkdir(parents=True, exist_ok=True)
V2_BF16_PATH = Path("/content/drive/MyDrive/FastFrames_QwenImage21_v02/checkpoints/qwen_image_2.1_pixel_head_v02_bf16.safetensors")

head_v1 = load_qwen21_pixel_head(V1_HEAD_PATH, device=DEVICE, dtype=torch.bfloat16)
head_v2 = load_qwen21_pixel_head(V2_BF16_PATH, device=DEVICE, dtype=torch.bfloat16)
vae_ref = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))
lp_fn = lpips.LPIPS(net="vgg", verbose=False).to(DEVICE).eval()


def edge_energy(rgb01: torch.Tensor) -> float:
    g = 0.299 * rgb01[..., 0] + 0.587 * rgb01[..., 1] + 0.114 * rgb01[..., 2]
    dx = (g[:, 1:] - g[:, :-1]).abs().mean()
    dy = (g[1:, :] - g[:-1, :]).abs().mean()
    return float((dx + dy).item())


metrics_v1, metrics_v2 = [], []
by_dom_v1, by_dom_v2 = {}, {}
t_vae_ms, t_v1_ms, t_v2_ms = [], [], []
crops_by_dom = {}

for r in traj_test:
    item = torch.load(CACHE_V2 / "onpolicy_1024" / r["file"], map_location="cpu", weights_only=True)
    f_in = item["f"].unsqueeze(0).to(DEVICE, torch.bfloat16)
    x_in = item["x"].unsqueeze(0).to(DEVICE, torch.bfloat16)
    sig = float(item["sigma"])
    ref_01 = ((item["tgt"][:3].float().permute(1, 2, 0) + 1.0) * 0.5).clamp(0, 1)

    # Time v0.1
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        p1 = head_v1(f_in, x_in)
    torch.cuda.synchronize()
    t_v1_ms.append((time.perf_counter() - t0) * 1000.0)
    v1_01 = ((p1[0, :3].float().cpu().permute(1, 2, 0) + 1.0) * 0.5).clamp(0, 1)

    # Time v0.2
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        p2 = head_v2(f_in, x_in, sigma=sig)
    torch.cuda.synchronize()
    t_v2_ms.append((time.perf_counter() - t0) * 1000.0)
    v2_01 = ((p2[0, :3].float().cpu().permute(1, 2, 0) + 1.0) * 0.5).clamp(0, 1)

    ref_m11 = (ref_01.permute(2, 0, 1).unsqueeze(0).to(DEVICE) * 2.0 - 1.0)
    v1_m11 = (v1_01.permute(2, 0, 1).unsqueeze(0).to(DEVICE) * 2.0 - 1.0)
    v2_m11 = (v2_01.permute(2, 0, 1).unsqueeze(0).to(DEVICE) * 2.0 - 1.0)

    psnr1 = -10.0 * math.log10(max(1e-10, float(((v1_01 - ref_01) ** 2).mean().item())))
    psnr2 = -10.0 * math.log10(max(1e-10, float(((v2_01 - ref_01) ** 2).mean().item())))
    with torch.no_grad():
        lp1 = float(lp_fn(v1_m11, ref_m11).mean().item())
        lp2 = float(lp_fn(v2_m11, ref_m11).mean().item())

    sh_ref = max(1e-6, edge_energy(ref_01))
    sh1 = 100.0 * edge_energy(v1_01) / sh_ref
    sh2 = 100.0 * edge_energy(v2_01) / sh_ref

    metrics_v1.append((psnr1, lp1, sh1))
    metrics_v2.append((psnr2, lp2, sh2))
    dom = r["domain"]
    by_dom_v1.setdefault(dom, []).append((psnr1, lp1, sh1))
    by_dom_v2.setdefault(dom, []).append((psnr2, lp2, sh2))

    if len(crops_by_dom.get(dom, [])) < 2:
        # Pick highest-gradient 384x384 detail region (automatically focuses on ticket text / eyes!)
        g_map = (ref_01[1:, :-1] - ref_01[:-1, :-1]).abs().mean(-1) + (ref_01[:-1, 1:] - ref_01[:-1, :-1]).abs().mean(-1)
        best_score_patch, by0, bx0 = -1.0, 320, 320
        for yy in range(64, 1024 - 448, 96):
            for xx in range(64, 1024 - 448, 96):
                sc = float(g_map[yy:yy+384, xx:xx+384].mean().item())
                if sc > best_score_patch:
                    best_score_patch, by0, bx0 = sc, yy, xx
        crops_by_dom.setdefault(dom, []).append((
            ref_01[by0:by0+384, bx0:bx0+384],
            v1_01[by0:by0+384, bx0:bx0+384],
            v2_01[by0:by0+384, bx0:bx0+384],
            psnr1, lp1, psnr2, lp2,
        ))

# Time Official 644MB VAE on 1024x1024 on A100
dummy_lat = torch.randn(1, 64, 64, 64)
for _ in range(5):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = vae_ref.decode(dummy_lat)
    torch.cuda.synchronize()
    t_vae_ms.append((time.perf_counter() - t0) * 1000.0)

med_vae = sorted(t_vae_ms)[len(t_vae_ms) // 2]
med_v1 = sorted(t_v1_ms)[len(t_v1_ms) // 2]
med_v2 = sorted(t_v2_ms)[len(t_v2_ms) // 2]

avg = lambda arr, idx: sum(x[idx] for x in arr) / len(arr)
v2_mb = V2_BF16_PATH.stat().st_size / (1024 * 1024)

md_lines = [
    "# Qwen-Image 2.1 PixelHead v0.2 (A100) vs v0.1 vs Official VAE — 1024×1024 Held-Out Report",
    "",
    "| Decoder | BF16 Size | 1024² Latency (A100) | Speedup | PSNR (↑ dB) | LPIPS-VGG (↓) | Edge Sharpness Ratio |",
    "| :--- | :---: | :---: | :---: | :---: | :---: | :---: |",
    f"| **Official Qwen-Image 2.1 VAE** | `644.2 MiB` | `{med_vae:.2f} ms` | `1.0×` | *(Ref)* | `0.0000` | `100.0%` |",
    f"| `PixelHead v0.1` (8×8 Linear, MSE) | `48.8 MiB` | `{med_v1:.2f} ms` | `{med_vae/med_v1:.1f}×` | `{avg(metrics_v1,0):.2f} dB` | `{avg(metrics_v1,1):.4f}` | `{avg(metrics_v1,2):.1f}%` |",
    f"| **`PixelHead v0.2` (1×1 Full-Res + LPIPS/FFT)** | **`{v2_mb:.1f} MiB`** | **`{med_v2:.2f} ms`** | **`{med_vae/med_v2:.1f}×`** | **`{avg(metrics_v2,0):.2f} dB`** | **`{avg(metrics_v2,1):.4f}`** | **`{avg(metrics_v2,2):.1f}%`** |",
    "",
    "## Per-Domain Sharpness Breakdown (`v0.1` → `v0.2`)",
    "",
    "| Domain | v0.1 PSNR / LPIPS / Sharpness | **v0.2 PSNR / LPIPS / Sharpness** |",
    "| :--- | :---: | :---: |",
]
for dom in sorted(by_dom_v2.keys()):
    a1, a2 = by_dom_v1[dom], by_dom_v2[dom]
    md_lines.append(
        f"| `{dom}` | `{avg(a1,0):.2f} dB` / `{avg(a1,1):.4f}` / `{avg(a1,2):.1f}%` | "
        f"**`{avg(a2,0):.2f} dB` / `{avg(a2,1):.4f}` / `{avg(a2,2):.1f}%`** |"
    )

report_txt = "\n".join(md_lines) + "\n"
(EVAL_V2 / "BENCHMARK_V02_VS_V01.md").write_text(report_txt, encoding="utf-8")

# Render 1:1 3-column comparison board (Official VAE | v0.1 | v0.2)
all_rows = []
for dom in ("small_text", "portrait", "fine_art"):
    for item_c in crops_by_dom.get(dom, []):
        all_rows.append((dom, *item_c))

cell_px = 384
board = Image.new("RGB", (cell_px * 3, cell_px * len(all_rows)), (18, 22, 30))
dr_b = ImageDraw.Draw(board)
for ri, (dom, c_ref, c_v1, c_v2, p1_d, l1_d, p2_d, l2_d) in enumerate(all_rows):
    panels = [
        (f"[{dom}] Official 644MB VAE", c_ref),
        (f"v0.1 ({p1_d:.1f}dB, LPIPS {l1_d:.3f})", c_v1),
        (f"v0.2 A100 ({p2_d:.1f}dB, LPIPS {l2_d:.3f})", c_v2),
    ]
    for ci, (lbl, patch) in enumerate(panels):
        pim = Image.fromarray((patch.numpy() * 255.0).round().astype(np.uint8))
        board.paste(pim, (ci * cell_px, ri * cell_px))
        dr_b.rectangle([ci * cell_px, ri * cell_px, (ci + 1) * cell_px - 1, ri * cell_px + 32], fill=(15, 23, 42))
        dr_b.text((ci * cell_px + 10, ri * cell_px + 8), lbl, fill=(255, 255, 255))

comp_png = EVAL_V2 / "v02_vs_v01_1to1_comparison.png"
board.save(comp_png)
display(Markdown(report_txt))
display(IPyImage(filename=str(comp_png)))
print(f"✅ 评测报告与 1:1 三方对比大图已保存至: {EVAL_V2}")